# **General Remarks**
This is the Python code developed for the article titled "Optimising Selective Maintenance for Marine Renewable Energy Systems Under Metocean Condition Uncertainty" (Noussis et al., 2026). The manuscript for this article has been submitted to the International Journal of Production Research for publication.

This Python code was developed and tested in Google Colab while using the v6e-1 TPU hardware accelerator. As such, certain aspects of the code may need to be revised if running it in other environments. The hardware accelarator used should not affect the ability for the code to run (just ensure there is at least one thread available for the multiprocessing blocks), but will heavily dictate the speed at which the code solves the given problem. The v6e-1 TPU accelerator is not only able to perform computations faster than traditional CPUs but is also able to provide over 40 CPU threads for multiprocessing.

Noussis, A., Saif, A., Khatab, A., & Diallo, C. (2026). Optimising selective maintenance for marine renewable energy under metocean condition uncertainty [Manuscript submitted for publication]. Department of Industrial Engineering, Dalhousie University.

# **Universal Code**

## *Library Loading*
NOTE: Gurobi API parameters have been omitted. However, the code should be fully functional if you provide your own parameters.

In [ ]:
# Loading relevant libraries
import itertools
import copy
import bisect
import math
import numpy
import scipy
import time
import multiprocessing
import pandas
from google.colab import files
from collections import defaultdict
import ast
import matplotlib.pyplot as plt
import seaborn
!pip install openpyxl
import openpyxl
!pip install gurobipy
from gurobipy import *

# Parameters for Gurobi API
params = {
"WLSACCESSID": 'OMITTED',
"WLSSECRET": 'OMITTED',
"LICENSEID": OMITTED,
}
env = Env(params=params)
env.setParam("LogToConsole", 0)

## *Generic Functions of Plan Generation*

In [ ]:
# Defining function for generating list of maintenance actions available for each component in subsystem i
# Inputs: subsystem under consideration
# Outputs: list of lists containing maintenance actions available for each component in subsystem under consideration
def avail_maint_list(i):
  # Initializing list for tracking maintenance actions available for each component
  Avail_Maint = []

  # For each component in subsystem i
  for j in range(N[i]):
    # Generating list of all possible maintenance actions for component
    Comp_Maint = list(range(0, L))

    # If component is functioning
    if Component_Status[i][j] == 1:
      # Removing minimal repair action
      Comp_Maint.pop(1)

    # Appending list of maintenance actions for component to list for subsystem
    Avail_Maint.append(Comp_Maint)

  # Returning outputs
  return Avail_Maint



# Defining function for enumerating all maintenance plan solutions to consider possible based on maintenance actions available for given subsystem
# Inputs: list of maintenance levels under consideration for each component in given subsystem
# Outputs: list of potential maintenance plans for subsystem under consideration
def solution_enumeration(maint_actions_list):
  # Generating all possible maintenance plans for given subsystem
  all_solutions_tuples = itertools.product(*maint_actions_list)
  # Converting maintenance plans to list format
  all_solutions_list = [list(permutation) for permutation in all_solutions_tuples]
  # Returning outputs
  return all_solutions_list

## *Calculating Conditional Reliability*

In [ ]:
# Defining the generalized conditional system reliability equation for a specified maintenance plan
# Inputs: 1) time t reliability is being calculated for, 2) maintenance plan (solution) under consideration
# Outputs: conditional reliability EQUATION for total system under specified maintenance plan
def rel_calc(t, maint_plan):
  # Initializing system conditional reliability (R^c(t))
  R = 1

  # For each subsystem
  for i in range(M):
    # Initializing subsystem reliability (R_i^c(t))
    R_i = 1

    # For each component in subsystem i
    for j in range(N[i]):
      # Calculating effective age and opertaional status of component E_ij after repairs based on selected maintenance plan (A_ij and V_ij)
      A_ij = Component_Age[i][j]*Age_Reduc_Coeff[maint_plan[i][j]]
      V_ij = 0

      # If component E_ij was functional before maintenance or has received a maintenance action other than "Do Nothing"
      if (Component_Status[i][j] == 1) or (maint_plan[i][j] != 0):
        # Setting component operational status to 1 (functional)
        V_ij = 1

      # Updating subsystem i conditional reliability by incorporating reliability of component E_ij (R_ij^c(t))
      R_i = R_i*(1 - V_ij*math.exp(-((A_ij + t)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j])/math.exp(-((A_ij)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j]))

    # Updating system reliability by incorporating reliability of subsystem i
    R_i = 1 - R_i
    R = R*R_i

  # Returning outputs
  return R

## *Experimental Instance Parameters*

In [ ]:
# Declaring relevant parameters for experimental instance
# Number of subsystems (M)
M = 5
# Number of components in each subsystem (N_i)
N = [4, 2, 3, 2, 3]
# Effective age of each component at beginning of maintenance break (B_ij)
Component_Age = [[60, 97, 68, 105],
                 [77, 71],
                 [100, 72, 73],
                 [103, 74],
                 [77, 104, 113]]
# Operational status of each component at beginning of maintenance break (S_ij)
Component_Status = [[1, 0, 0, 0],
                    [0, 1],
                    [1, 0, 1],
                    [0, 0],
                    [0, 1, 0]]
# Scale parameters for component Weibull distributions (beta_ij)
Component_Weibull_eta = [[100, 100, 100, 100],
                         [160, 160],
                         [120, 120, 120],
                         [187, 187],
                         [80, 80, 80]]

# Shape parameters for component Weibull distributions (eta_ij)
Component_Weibull_beta = [[1.5, 1.5, 1.5, 1.5],
                          [2.4, 2.4],
                          [1.8, 1.8, 1.8],
                          [2.8, 2.8],
                          [1.2, 1.2, 1.2]]



# Number of maintenance levels under consideration for every component (L_ij)
L = 4
# Age reduction coefficient for maintenance level
Age_Reduc_Coeff = [1, 1, 0.25, 0]
# Maximum break duration (T_0)
T0 = 120
# Maintenance budget available (C_0)
C0 = 500
# Minimum reliability required (R_0)
R0 = 0.5
# Length of upcoming mission (W)
W = 60
# Variable cost per unit of time spent performing maintenance (c_v)
c_labor = 5
# Penalty cost per unit of system downtime (c_d)
c_downtime = 33
# Energy production rate function (epsilon(t))
energy_rate = 0.5
# Probability of maintenance occurring onshore (p)
p = 0.5
# Onshore maintenance cost factor (k_c)
k_cost = 1.25
# Onshore maintenance time factor (k_t)
k_time = 1.5



# Duration of corrective and preventive maintenance actions for each component while offshore (t_ijl^cm and t_ijl^pm)
# First dimension is subsystem, second is component, third is whether it's CM (index 0) or PM (index 1), fourth is level of maintenance
# This list must be adjusted based on the number of subsystems, number of components in each subsystem, and number of levels of maintenance
T_site = [[[[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]]],
          [[[0.00, 3.60, 5.40, 10.80],[0.00, 0.00, 2.70, 5.40]], [[0.00, 3.60, 5.40, 10.80],[0.00, 0.00, 2.70, 5.40]]],
          [[[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]]],
          [[[0.00, 4.00, 6.00, 12.00],[0.00, 0.00, 3.00, 6.00]], [[0.00, 4.00, 6.00, 12.00],[0.00, 0.00, 3.00, 6.00]]],
          [[[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]], [[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]], [[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]]]]
# Fixed cost of corrective and preventive maintenance actions for each component while offshore (c_ijl^cm and c_ijl^pm)
# Same structure as T_site
C_site = [[[[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]]],
          [[[0.00, 15.00, 22.50, 45.00],[0.00, 0.00, 11.25, 22.50]], [[0.00, 15.00, 22.50, 45.00],[0.00, 0.00, 11.25, 22.50]]],
          [[[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]]],
          [[[0.00, 16.67, 25.00, 50.00],[0.00, 0.00, 12.50, 25.00]], [[0.00, 16.67, 25.00, 50.00],[0.00, 0.00, 12.50, 25.00]]],
          [[[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]], [[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]], [[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]]]]



# Creating copy of offshore maintenance cost and duration data
C_expected = copy.deepcopy(C_site)
T_expected = copy.deepcopy(T_site)
# For each subsystem
for i in range(M):
  # For each component in subsystem i
  for j in range(N[i]):
    # For both lists storing corrective and preventive maintenance data
    for k in range(2):
      # Calculating expected cost and duration of each maintenance action (C_ijl and T_ijl) based on onshore cost and time factors (k_c and k_t) and probability of onshore work (p)
      C_expected[i][j][k] = [l*(k_cost*p + 1 - p) for l in C_expected[i][j][k]]
      T_expected[i][j][k] = [l*(k_time*p + 1 - p) for l in T_expected[i][j][k]]

# **Deterministic Variant**

## *Subsystem Plan Generation Function*

In [ ]:
# Defining function for generating complete, sorted, culled list of maintenance plans for each subsystem
# Inputs: none
# Outputs: 1) complete, sorted list of feasible maintenance plans for each subsystem, as well as sorted lists reporting each plan's respective 2) duration, 3) maintenance cost,
# 4) subsystem conditional reliability, and 5) total cost (maintenance + labour)
def subsystem_solution_generator_tcr():
  # Initializing lists for storing subsystem maintenance plans (Lambda_i) and respective durations (T_i^lambda), repair costs, conditional reliabilities (R_i^clambda(W)), and total costs (C_i^lambda)
  all_subsys_solns = []
  all_subsys_soln_durations = []
  all_subsys_soln_costs = []
  all_subsys_soln_reliabilities = []
  all_subsys_soln_TOTAL_costs = []

  # For each subsystem
  for i in range(M):
    # Generating list of available maintenance actions for each component in subsystem i
    Avail_Maint = avail_maint_list(i)
    # Generating list of all possible maintenance plans for subsystem i
    subsys_solns = solution_enumeration(Avail_Maint)
    # Initializing lists for tracking maintenance plan data for subsystem i
    subsys_soln_durations = []
    subsys_soln_costs = []
    subsys_soln_reliabilities = []
    subsys_soln_TOTAL_costs = []
    # Initializing counter for scanning through potential subsystem i maintenance plans
    s = 0

    # While still not done scanning through subsystem i maintenance plans
    while s < len(subsys_solns):
      # Initializing subsystem i maintenance plan duration, repair cost, and conditional failure
      Subsys_Soln_Duration = 0
      Subsys_Soln_Cost = 0
      Subsys_Soln_Failure = 1

      # For each component in subsystem i
      for j in range(N[i]):
        # Adding component E_ij maintenance action duration (T_ijl) and repair cost to total subsystem i plan duration and repair cost
        Subsys_Soln_Duration += T_expected[i][j][Component_Status[i][j]][subsys_solns[s][j]]
        Subsys_Soln_Cost += C_expected[i][j][Component_Status[i][j]][subsys_solns[s][j]]
        # Calculating updated component E_ij effective age and operational status (A_ij and V_ij) based on selected maintenance action
        A_ij = Component_Age[i][j]*Age_Reduc_Coeff[subsys_solns[s][j]]
        V_ij = 0

        # If component E_ij was functional before maintenance or has received a maintenance action other than "Do Nothing"
        if (Component_Status[i][j] == 1) or (subsys_solns[s][j] != 0):
          # Setting component operational status to 1 (functional)
          V_ij = 1

        # Calculating component E_ij conditional reliability (R_ij^c(W)) based on selected maintenance action
        R_ij = V_ij*math.exp(-((A_ij + W)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j])/math.exp(-((A_ij)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j])
        # Factoring in component E_ij conditional reliability to subsystem i maintenance plan conditional failure (1 - R_i^c(W))
        Subsys_Soln_Failure = Subsys_Soln_Failure*(1 - R_ij)

      # Calculating subsystem i conditional reliability according to conditional failure of plan (R_i^c(W))
      Subsys_Soln_Reliability = 1 - Subsys_Soln_Failure

      # If subsystem i maintenance plan duration exceeds break duration, total cost exceeds maintenance budget, or conditional reliability is below minimum threshold
      if Subsys_Soln_Duration > T0 or (Subsys_Soln_Cost + Subsys_Soln_Duration*c_labor) > C0 or Subsys_Soln_Reliability < R0:
        # Removing subsystem i maintenance plan
        subsys_solns.pop(s)
        # Decrementing counter for scanning through subsystem i maintenance plans since a plan has been removed
        s -= 1
      # Else
      else:
        # Appending plan's duration, repair cost, conditional reliability, and total cost to overall lists for subsystem i maintenance plans
        subsys_soln_durations.append(Subsys_Soln_Duration)
        subsys_soln_costs.append(Subsys_Soln_Cost)
        subsys_soln_reliabilities.append(Subsys_Soln_Reliability)
        subsys_soln_TOTAL_costs.append(Subsys_Soln_Cost + Subsys_Soln_Duration*c_labor)

      # Incrementing counter for scanning through subsystem i maintenance plans to move onto next plan
      s += 1

    # If no feasible subsystem plans were found
    if len(subsys_solns) == 0:
      # Reporting that there are no feasible solutions to the problem based on its current parameters
      print("No feasible solutions based on given parameters.")
      # Appending empty lists of feasible subsystem i maintenance plans and associated data to overall lists
      all_subsys_solns.append(subsys_solns)
      all_subsys_soln_durations.append(subsys_soln_durations)
      all_subsys_soln_costs.append(subsys_soln_costs)
      all_subsys_soln_reliabilities.append(subsys_soln_reliabilities)
      all_subsys_soln_TOTAL_costs.append(subsys_soln_TOTAL_costs)
      # Ending subsystem maintenance plan generation
      break

    # Sorting feasible subsystem i maintenance plans based on durations in increasing order
    subsys_soln_durations, subsys_solns, subsys_soln_costs, subsys_soln_reliabilities, subsys_soln_TOTAL_costs = zip(*sorted(zip(subsys_soln_durations, subsys_solns, subsys_soln_costs, subsys_soln_reliabilities, subsys_soln_TOTAL_costs)))
    # Converting sorted lists of subsystem i maintenance plans and associated data from tuples to lists
    subsys_solns = list(subsys_solns)
    subsys_soln_durations = list(subsys_soln_durations)
    subsys_soln_costs = list(subsys_soln_costs)
    subsys_soln_reliabilities = list(subsys_soln_reliabilities)
    subsys_soln_TOTAL_costs = list(subsys_soln_TOTAL_costs)
    # Appending lists of feasible, sorted subsystem i maintenance plans and associated data to overall list (Lambda_i, T_i^lambda, R_i^clambda(W), C_i^lambda)
    all_subsys_solns.append(subsys_solns)
    all_subsys_soln_durations.append(subsys_soln_durations)
    all_subsys_soln_costs.append(subsys_soln_costs)
    all_subsys_soln_reliabilities.append(subsys_soln_reliabilities)
    all_subsys_soln_TOTAL_costs.append(subsys_soln_TOTAL_costs)

  # Returning outputs
  return all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, all_subsys_soln_TOTAL_costs

## *Recursive Plan Builder Function*

In [ ]:
# Defining subfunction for recursive construction of complete maintenance plans based on current partial maintenance plan being considered
# Inputs: 1) complete list of all feasible subsystem solutions and their associated 2) durations, 3) repair costs, 4) conditional reliabilities, and 5) total costs,
# 6) the current partial maintenance plan being considered and its associated 7) duration, 8) repair cost, 9) conditional reliability, and 10) total cost, and
# 11) a complete list of all maintenance plans for the current subsystem under consideration that are feasible according to the partial plan that's been constructed and their associated 12) durations,
# 13) repair costs, 14) conditional reliabilities, and 15) total costs, along with 16) a variable tracking the current subsystem under consideration (m)
# Outputs: 1) list of feasible, complete maintenance plans and their associated 2) durations, 3) repair costs, and 4) conditional system reliabilities
def plan_builder_tcr(all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, current_plan, current_plan_duration, current_plan_cost, current_plan_reliability, current_subsys_plans, current_subsys_plan_durations, current_subsys_plan_costs, current_subsys_plan_reliabilities, m, all_subsys_soln_TOTAL_costs, current_plan_TOTAL_cost, current_subsys_plan_TOTAL_costs):
  # Incrementing the variable tracking the current subsystem being considered for building maintenance plans
  m += 1
  # Initializing lists for storing feasible complete maintenance plans and associated data
  built_plans = []
  built_plan_durations = []
  built_plan_costs = []
  built_plan_reliabilities = []
  # Initializing lists for storing feasible maintenance plans and associated data for next subsystem to be considered
  next_subsys_plans = []
  next_subsys_plan_durations = []
  next_subsys_plan_costs = []
  next_subsys_plan_reliabilities = []
  next_subsys_plan_TOTAL_costs = []

  # If the current subsystem under consideration is not the final subsystem
  if m < M:
    # Copying all potential plans and associated data for the next subsystem to be considered (Lambda_i+1^timecopy)
    next_subsys_plans = copy.deepcopy(all_subsys_solns[m])
    next_subsys_plan_durations = copy.deepcopy(all_subsys_soln_durations[m])
    next_subsys_plan_costs = copy.deepcopy(all_subsys_soln_costs[m])
    next_subsys_plan_reliabilities = copy.deepcopy(all_subsys_soln_reliabilities[m])
    next_subsys_plan_TOTAL_costs = copy.deepcopy(all_subsys_soln_TOTAL_costs[m])

  # For each potential maintenance plan for the CURRENT subsystem being considered (Lambda_i)
  for soln in range(len(current_subsys_plans)):
    # Copying the current, partial maintenance plan that has been built (Plan_current)
    next_plan = copy.deepcopy(current_plan)
    # Appending the potential maintenance plan for the current subsystem being considered to the current, partial maintenance plan (Plan_next = Plan_current + Subsystem i Plan lambda)
    next_plan.append(current_subsys_plans[soln])
    # Adding the duration of the subsystem plan under consideration to the current, partial maintenance plan's (T_next = T_current + T_i^lambda)
    next_plan_duration = current_plan_duration + current_subsys_plan_durations[soln]
    # Adding the repair cost of the subsystem plan under consideration to the current, partial maintenance plan's
    next_plan_cost = current_plan_cost + current_subsys_plan_costs[soln]
    # Multiplying the conditional reliability of the subsystem plan under consideration to the current, partial maintenance plan's (R_next = R_current * R_ci^lambda(W))
    next_plan_reliability = current_plan_reliability*current_subsys_plan_reliabilities[soln]
    # Adding the total cost of the subsystem plan under consideration to the current, partial maintenance plan's (C_next = C_current + C_i^lambda)
    next_plan_TOTAL_cost = current_plan_TOTAL_cost + current_subsys_plan_TOTAL_costs[soln]

    # If currently considering subsystem plans for the final subsystem
    if m == M:
      # Storing the complete maintenance plan and associated data (theta)
      built_plans.append(next_plan)
      built_plan_durations.append(next_plan_duration)
      built_plan_costs.append(next_plan_cost)
      built_plan_reliabilities.append(next_plan_reliability)
    # Else
    else:
      ###### Break Duration Constraint ######
      # Calculating how much time remains to be allocated to remaining subsystem maintenance plans (T_0 - T_next)
      remaining_duration = T0 - next_plan_duration
      # Finding the index of the first infeasible maintenance plan in the list of plans for the next subsystem to be considered based on the remaining time
      # (NOTE: the lists are already sorted from lowest to highest durations)
      first_infeasible_next_ss_plan = bisect.bisect(next_subsys_plan_durations, remaining_duration)
      # Removing all subsystem plans and associated data with an index equal to or higher than the first infeasible plan's
      next_subsys_plans = next_subsys_plans[:first_infeasible_next_ss_plan]
      next_subsys_plan_durations = next_subsys_plan_durations[:first_infeasible_next_ss_plan]
      next_subsys_plan_costs = next_subsys_plan_costs[:first_infeasible_next_ss_plan]
      next_subsys_plan_reliabilities = next_subsys_plan_reliabilities[:first_infeasible_next_ss_plan]
      next_subsys_plan_TOTAL_costs = next_subsys_plan_TOTAL_costs[:first_infeasible_next_ss_plan]

      # If there is at least one subsystem plan that has remained (is feasible based on the current partial plan)
      if(len(next_subsys_plans) > 0):
        ###### Maintenance Budget Constraint ######
        # Copying the remaining feasible plans and associated data for the next subsystem to be considered (Lambda_i+1^copy)
        next_subsys_plans_costrelcopy = copy.deepcopy(next_subsys_plans)
        next_subsys_plan_durations_costrelcopy = copy.deepcopy(next_subsys_plan_durations)
        next_subsys_plan_costs_costrelcopy = copy.deepcopy(next_subsys_plan_costs)
        next_subsys_plan_reliabilities_costrelcopy = copy.deepcopy(next_subsys_plan_reliabilities)
        next_subsys_plan_TOTAL_costs_costrelcopy = copy.deepcopy(next_subsys_plan_TOTAL_costs)
        # Re-sorting lists of feasible plans and associated data for the next subsystem to be considered to be increasing based on total costs
        next_subsys_plan_TOTAL_costs_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_durations_costrelcopy, next_subsys_plan_reliabilities_costrelcopy = zip(*sorted(zip(next_subsys_plan_TOTAL_costs_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_durations_costrelcopy, next_subsys_plan_reliabilities_costrelcopy)))
        next_subsys_plans_costrelcopy = list(next_subsys_plans_costrelcopy)
        next_subsys_plan_durations_costrelcopy = list(next_subsys_plan_durations_costrelcopy)
        next_subsys_plan_costs_costrelcopy = list(next_subsys_plan_costs_costrelcopy)
        next_subsys_plan_reliabilities_costrelcopy = list(next_subsys_plan_reliabilities_costrelcopy)
        next_subsys_plan_TOTAL_costs_costrelcopy = list(next_subsys_plan_TOTAL_costs_costrelcopy)
        # Calculating how much budget remains to be allocated to remaining subsystem maintenance plans (C_0 - C_next)
        remaining_budget = C0 - next_plan_TOTAL_cost
        # Finding the index of the first infeasible maintenance plan in the list of plans for the next subsystem to be considered based on the remaining budget
        first_infeasible_next_ss_plan = bisect.bisect(next_subsys_plan_TOTAL_costs_costrelcopy, remaining_budget)
        # Removing all subsystem plans and associated data with an index equal to or higher than the first infeasible plan's
        next_subsys_plans_costrelcopy = next_subsys_plans_costrelcopy[:first_infeasible_next_ss_plan]
        next_subsys_plan_durations_costrelcopy = next_subsys_plan_durations_costrelcopy[:first_infeasible_next_ss_plan]
        next_subsys_plan_costs_costrelcopy = next_subsys_plan_costs_costrelcopy[:first_infeasible_next_ss_plan]
        next_subsys_plan_reliabilities_costrelcopy = next_subsys_plan_reliabilities_costrelcopy[:first_infeasible_next_ss_plan]
        next_subsys_plan_TOTAL_costs_costrelcopy = next_subsys_plan_TOTAL_costs_costrelcopy[:first_infeasible_next_ss_plan]

        # If there is at least one subsystem plan that has remained (is feasible based on the current partial plan)
        if(len(next_subsys_plans_costrelcopy) > 0):
          ###### Minimum Reliability Constraint ######
          # Re-sorting lists of feasible plans and associated data for the next subsystem to be considered to be increasing based on conditional reliabilities
          next_subsys_plan_reliabilities_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_durations_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plan_TOTAL_costs_costrelcopy = zip(*sorted(zip(next_subsys_plan_reliabilities_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_durations_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plan_TOTAL_costs_costrelcopy)))
          next_subsys_plans_costrelcopy = list(next_subsys_plans_costrelcopy)
          next_subsys_plan_durations_costrelcopy = list(next_subsys_plan_durations_costrelcopy)
          next_subsys_plan_costs_costrelcopy = list(next_subsys_plan_costs_costrelcopy)
          next_subsys_plan_reliabilities_costrelcopy = list(next_subsys_plan_reliabilities_costrelcopy)
          next_subsys_plan_TOTAL_costs_costrelcopy = list(next_subsys_plan_TOTAL_costs_costrelcopy)
          # Calculating minimum conditional reliabilty that must be met by each remaining subsystem (R_0 - R_next)
          minimum_reliability = R0/next_plan_reliability
          # Finding the index of the last infeasible maintenance plan in the list of plans for the next subsystem to be considered based on the minimum conditional reliability
          first_infeasible_next_ss_plan = bisect.bisect_left(next_subsys_plan_reliabilities_costrelcopy, minimum_reliability)
          # Removing all subsystem plans and associated data with an index equal to or less than the last infeasible plan's
          next_subsys_plans_costrelcopy = next_subsys_plans_costrelcopy[first_infeasible_next_ss_plan:]
          next_subsys_plan_durations_costrelcopy = next_subsys_plan_durations_costrelcopy[first_infeasible_next_ss_plan:]
          next_subsys_plan_costs_costrelcopy = next_subsys_plan_costs_costrelcopy[first_infeasible_next_ss_plan:]
          next_subsys_plan_reliabilities_costrelcopy = next_subsys_plan_reliabilities_costrelcopy[first_infeasible_next_ss_plan:]
          next_subsys_plan_TOTAL_costs_costrelcopy = next_subsys_plan_TOTAL_costs_costrelcopy[first_infeasible_next_ss_plan:]

          # If there is at least one subsystem plan that has remained (is feasible based on the current partial plan)
          if len(next_subsys_plans_costrelcopy) > 0:
            # Re-sorting lists of feasible plans and associated data for the next subsystem to be considered to be increasing based on total durations
            next_subsys_plan_durations_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_reliabilities_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plan_TOTAL_costs_costrelcopy = zip(*sorted(zip(next_subsys_plan_durations_costrelcopy, next_subsys_plans_costrelcopy, next_subsys_plan_reliabilities_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plan_TOTAL_costs_costrelcopy)))
            next_subsys_plans_costrelcopy = list(next_subsys_plans_costrelcopy)
            next_subsys_plan_durations_costrelcopy = list(next_subsys_plan_durations_costrelcopy)
            next_subsys_plan_costs_costrelcopy = list(next_subsys_plan_costs_costrelcopy)
            next_subsys_plan_reliabilities_costrelcopy = list(next_subsys_plan_reliabilities_costrelcopy)
            next_subsys_plan_TOTAL_costs_costrelcopy = list(next_subsys_plan_TOTAL_costs_costrelcopy)
            # Recursively calling this function with updated partial maintenance plan and remaining feasible plans for next subsystem
            set_of_built_plans, set_of_built_plan_durations, set_of_built_plan_costs, set_of_built_plan_reliabilities = plan_builder_tcr(all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, next_plan, next_plan_duration, next_plan_cost, next_plan_reliability, next_subsys_plans_costrelcopy, next_subsys_plan_durations_costrelcopy, next_subsys_plan_costs_costrelcopy, next_subsys_plan_reliabilities_costrelcopy, m, all_subsys_soln_TOTAL_costs, next_plan_TOTAL_cost, next_subsys_plan_TOTAL_costs_costrelcopy)
            # Storing the list of complete maintenance plans and associated returned from the above recursion
            built_plans += set_of_built_plans
            built_plan_durations += set_of_built_plan_durations
            built_plan_costs += set_of_built_plan_costs
            built_plan_reliabilities += set_of_built_plan_reliabilities

  # Returning outputs
  return built_plans, built_plan_durations, built_plan_costs, built_plan_reliabilities

## *Function for Generating Complete, Feasible Maintenance Plans*

In [ ]:
# Defining master function for constructing all feasible, complete maintenance plans
# Inputs: none
# Outputs: 1) list of all feasible, complete maintenance plans and their associated 2) durations, 3) repair costs, 4) conditional system reliabilities, and 5) conditional subsystem reliabilities
def plan_list_generator_time_cost_reliability_constraints():
  # Initializing lists for storing all feasible, complete maintenance plans and associated data (Theta)
  complete_maint_plan_master_list = []
  complete_maint_plan_duration_master_list = []
  complete_maint_plan_cost_master_list = []
  complete_maint_plan_reliability_master_list = []
  complete_maint_plan_subsys_reliability_master_list = []

  # If the system only has one subsystem
  if M == 1:
    # Generating all feasible maintenance plans for the subsystem
    complete_maint_plans, complete_maint_plan_durations, complete_maint_plan_costs, complete_maint_plan_reliabilities, complete_maint_plan_TOTAL_costs = subsystem_solution_generator_tcr()
    # Storing all feasible maintenance plans and associated data for the subsystem as the complete maintenance plan results
    complete_maint_plan_master_list = complete_maint_plans[0]
    complete_maint_plan_duration_master_list = complete_maint_plan_durations[0]
    complete_maint_plan_cost_master_list = complete_maint_plan_costs[0]
    complete_maint_plan_reliability_master_list = complete_maint_plan_reliabilities[0]
  # Else
  else:
    # Generating all feasible maintenance plans and associated data for all subsystems
    all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, all_subsys_soln_TOTAL_costs = subsystem_solution_generator_tcr()

    # For each subsystem
    for i in range(len(all_subsys_solns)):
      # If there are no feasible plans for the subsystem

      if len(all_subsys_solns[i]) == 0:
        # Returning the empty lists of complete maintenance plans and associated data (i.e., there are no feasible solutions)
        return complete_maint_plan_master_list, complete_maint_plan_duration_master_list, complete_maint_plan_cost_master_list, complete_maint_plan_reliability_master_list, complete_maint_plan_subsys_reliability_master_list

    # Isolating the feasible maintenance plans and associated data for subsystem 1
    subsys_1_solns = all_subsys_solns[0]
    subsys_1_soln_durations = all_subsys_soln_durations[0]
    subsys_1_soln_costs = all_subsys_soln_costs[0]
    subsys_1_soln_reliabilities = all_subsys_soln_reliabilities[0]
    subsys_1_soln_TOTAL_costs = all_subsys_soln_TOTAL_costs[0]
    # Copying the feasible maintenance plans and associated data for subsystem 2
    subsys_2_solns = copy.deepcopy(all_subsys_solns[1])
    subsys_2_soln_durations = copy.deepcopy(all_subsys_soln_durations[1])
    subsys_2_soln_costs = copy.deepcopy(all_subsys_soln_costs[1])
    subsys_2_soln_reliabilities = copy.deepcopy(all_subsys_soln_reliabilities[1])
    subsys_2_soln_TOTAL_costs = copy.deepcopy(all_subsys_soln_TOTAL_costs[1])

    # For each potential maintenance plan for subsystem 1 (Lambda_1)
    for soln in range(len(subsys_1_solns)):
      # Initializing the variable tracking the current subsystem being considered for building maintenance plans
      m = 1
      # Using the potential maintenance plan and assoicated data for subsystem 1 as the basis for a complete maintenance plan (Plan_next, T_next, R_next, C_next)
      current_plan = [subsys_1_solns[soln]]
      current_plan_duration = subsys_1_soln_durations[soln]
      current_plan_cost = subsys_1_soln_costs[soln]
      current_plan_reliability = subsys_1_soln_reliabilities[soln]
      current_plan_TOTAL_cost = subsys_1_soln_TOTAL_costs[soln]
      ###### Break Duration Constraint ######
      # Calculating how much time remains to be allocated to remaining subsystem maintenance plans (T_0 - T_next)
      remaining_duration = T0 - current_plan_duration
      # Finding the index of the first infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the remaining time
      # (NOTE: the lists are already sorted from lowest to highest durations)
      first_infeasible_ss2_plan = bisect.bisect(subsys_2_soln_durations, remaining_duration)
      # Removing all subsystem 2 plans and associated data with an index equal to or higher than the first infeasible plan's
      subsys_2_solns = subsys_2_solns[:first_infeasible_ss2_plan]
      subsys_2_soln_durations = subsys_2_soln_durations[:first_infeasible_ss2_plan]
      subsys_2_soln_costs = subsys_2_soln_costs[:first_infeasible_ss2_plan]
      subsys_2_soln_reliabilities = subsys_2_soln_reliabilities[:first_infeasible_ss2_plan]
      subsys_2_soln_TOTAL_costs = subsys_2_soln_TOTAL_costs[:first_infeasible_ss2_plan]

      # If there is at least one subsystem 2 plan that has remained (is feasible based on the subsystem 1 plan that was selected)
      if(len(subsys_2_solns) > 0):
        ###### Maintenance Budget Constraint ######
        # Copying the remaining feasible plans and associated data for the next subsystem to be considered (Lambda_2^copy)
        subsys_2_solns_costrelcopy = copy.deepcopy(subsys_2_solns)
        subsys_2_soln_durations_costrelcopy = copy.deepcopy(subsys_2_soln_durations)
        subsys_2_soln_costs_costrelcopy = copy.deepcopy(subsys_2_soln_costs)
        subsys_2_soln_reliabilities_costrelcopy = copy.deepcopy(subsys_2_soln_reliabilities)
        subsys_2_soln_TOTAL_costs_costrelcopy = copy.deepcopy(subsys_2_soln_TOTAL_costs)
        # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on total costs
        subsys_2_soln_TOTAL_costs_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_reliabilities_costrelcopy = zip(*sorted(zip(subsys_2_soln_TOTAL_costs_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_reliabilities_costrelcopy)))
        subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
        subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
        subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
        subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
        subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
        # Calculating how much budget remains to be allocated to remaining subsystem maintenance plans (C_0 - C_next)
        remaining_budget = C0 - current_plan_TOTAL_cost
        # Finding the index of the first infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the remaining budget
        first_infeasible_next_ss_plan = bisect.bisect(subsys_2_soln_TOTAL_costs_costrelcopy, remaining_budget)
        # Removing all subsystem 2 plans and associated data with an index equal to or higher than the first infeasible plan's
        subsys_2_solns_costrelcopy = subsys_2_solns_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_durations_costrelcopy = subsys_2_soln_durations_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_costs_costrelcopy = subsys_2_soln_costs_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_reliabilities_costrelcopy = subsys_2_soln_reliabilities_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_TOTAL_costs_costrelcopy = subsys_2_soln_TOTAL_costs_costrelcopy[:first_infeasible_next_ss_plan]

        # If there is at least one subsystem 2 plan that has remained (is feasible based on the subsystem 1 plan that was selected)
        if(len(subsys_2_solns_costrelcopy) > 0):
          ###### Minimum Reliability Constraint ######
          # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on conditional reliabilities
          subsys_2_soln_reliabilities_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy = zip(*sorted(zip(subsys_2_soln_reliabilities_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy)))
          subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
          subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
          subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
          subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
          subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
          # Calculating minimum conditional reliabilty that must be met by each remaining subsystem (R_0 - R_next)
          minimum_reliability = R0/current_plan_reliability
          # Finding the index of the last infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the minimum conditional reliability
          first_infeasible_next_ss_plan = bisect.bisect_left(subsys_2_soln_reliabilities_costrelcopy, minimum_reliability)
          # Removing all subsystem 2 plans and associated data with an index equal to or less than the last infeasible plan's
          subsys_2_solns_costrelcopy = subsys_2_solns_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_durations_costrelcopy = subsys_2_soln_durations_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_costs_costrelcopy = subsys_2_soln_costs_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_reliabilities_costrelcopy = subsys_2_soln_reliabilities_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_TOTAL_costs_costrelcopy = subsys_2_soln_TOTAL_costs_costrelcopy[first_infeasible_next_ss_plan:]

          # If there is at least one subsystem 2 plan that has remained (is feasible based on the current partial plan)
          if len(subsys_2_solns_costrelcopy) > 0:
            # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on total durations
            subsys_2_soln_durations_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy = zip(*sorted(zip(subsys_2_soln_durations_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy)))
            subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
            subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
            subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
            subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
            subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
            # Calling the recursive plan builder subfunction with selected subsystem 1 plan and remaining feasible plans for subsystem 2
            complete_maint_plans, complete_maint_plan_durations, complete_maint_plan_costs, complete_maint_plan_reliabilities = plan_builder_tcr(all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, current_plan, current_plan_duration, current_plan_cost, current_plan_reliability, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, m, all_subsys_soln_TOTAL_costs, current_plan_TOTAL_cost, subsys_2_soln_TOTAL_costs_costrelcopy)

            # For all complete, feasible maintenance plans constructed from the plan builder subfunction
            for n in range(len(complete_maint_plans)):
              # Initializing list to store subsystem conditional reliabilities for selected plan
              current_maint_plan_subsys_reliabilities = []

              # For each subsystem
              for i in range(M):
                # Finding the index of the corresponding subsystem maintenance plan and storing the associated subsystem conditional reliability
                subsys_plan_index = all_subsys_solns[i].index(complete_maint_plans[n][i])
                current_maint_plan_subsys_reliabilities.append(all_subsys_soln_reliabilities[i][subsys_plan_index])

              # Appending the list of subsystem conditional reliabilities for the selected complete maintenance plan
              complete_maint_plan_subsys_reliability_master_list.append(current_maint_plan_subsys_reliabilities)

            # Storing the complete, feasible maintenance plans that were constructed and their associated data in the master lists
            complete_maint_plan_master_list += complete_maint_plans
            complete_maint_plan_duration_master_list += complete_maint_plan_durations
            complete_maint_plan_cost_master_list += complete_maint_plan_costs
            complete_maint_plan_reliability_master_list += complete_maint_plan_reliabilities

  # Returning outputs
  return complete_maint_plan_master_list, complete_maint_plan_duration_master_list, complete_maint_plan_cost_master_list, complete_maint_plan_reliability_master_list, complete_maint_plan_subsys_reliability_master_list

## *Uptime and LCOE Calculator*

In [ ]:
# Defining the function for calculating the expected uptime of the asset during a mission after executing a specific maintenance plan
# Inputs: relevant data for maintenance plan under consideration (1) maintenance plan under consideration, 2) expected time to perform maintenance plan, 3) expected repair cost to perform maintenance plan)
# Outputs: expected 1) uptime, 2) total cost of the selected plan, 3) energy production, 4) LCOE
def expected_uptime(Maint_Plan_Params):
  # Loading relevant data for selected maintenance plan (actions selected, total duration, total repair cost)
  maint_plan = Maint_Plan_Params[0]
  Maint_Time = Maint_Plan_Params[1]
  Maint_Cost = Maint_Plan_Params[2]
  # Calculating expected uptime of the asset according to the mission length and selected maintenance plan (U_theta(W))
  avg_uptime, throwaway = scipy.integrate.quad(rel_calc, 0, W, maint_plan)
  # Calculating the expected total cost of the selected maintenance plan (repair, labour, and downtime)
  Total_Cost = Maint_Cost + c_labor*Maint_Time + c_downtime*(W - avg_uptime)
  # Calculating the expected energy produced by the asset according to the expected uptime
  Energy_Prod = energy_rate*avg_uptime

  # If energy is produced (i.e., the device is functional when the mission begins)
  if Energy_Prod > 0:
    # Calculating expected LCOE of the asset according to the selected maintenance plan (zeta_theta)
    LCOE = Total_Cost/Energy_Prod
  # Else
  else:
    # Setting expected LCOE of the asset equal to infinity (division by zero)
    LCOE = numpy.inf

  # Returning outputs
  return [avg_uptime, Total_Cost, Energy_Prod, LCOE]

## *Deterministic MRE SMP Solution Methodology Master Function*

In [ ]:
# Defining master function for finding optimal maintenance plan
# Inputs: none
# Outputs: 1) number of feasible solutions, 2) computation time,
# For LCOE minimization: 3) optimal maintenance plan and associated 4) duration, 5) repair cost, 6) labour cost, 7) downtime cost, 8) total cost, 9) subsystem conditional reliabilities, 10) system conditional reliability, 11) expected uptime, 12) expected energy produced, and 13) expected LCOE
# For cost minimization: 14) optimal maintenance plan and associated 15) duration, 16) repair cost, 17) labour cost, 18) downtime cost, 19) total cost, 20) subsystem conditional reliabilities, 21) system conditional reliability, 22) expected uptime, 23) expected energy produced, and 24) expected LCOE
# For energy production maximization: 25) optimal maintenance plan and associated 26) duration, 27) repair cost, 28) labour cost, 29) downtime cost, 30) total cost, 31) subsystem conditional reliabilities, 32) system conditional reliability, 33) expected uptime, 34) expected energy produced, and 35) expected LCOE
def deterministic_variant_tcr():
  # Recording start time of experiment
  overall_start_time = time.time()
  print(f"Generating plans with time, cost, and reliability constraints (T_0 = {T0}, C_0 = {C0}, R_0 = {R0})...")
  # Recording start time of feasible plan generation
  soln_generation_start_time = time.time()
  # Generating all complete, feasible maintenance plans based on given parameters
  maint_plan_list, plan_duration_list, plan_cost_list, plan_reliability_list, plan_subsys_reliability_list = plan_list_generator_time_cost_reliability_constraints()
  # Calculating and reporting time spent generating maintenance plans
  soln_generation_end_time = time.time()
  print(f"{len(maint_plan_list)} plans generated!")
  print(f"Solution generation took {soln_generation_end_time - soln_generation_start_time} seconds.")

  # If no feasible maintenance plans were constructed
  if len(maint_plan_list) == 0:
    # Calculating total experiment time and reporting infeasibility
    overall_end_time = time.time()
    total_time = overall_end_time - overall_start_time
    print("No feasible solutions found.")
    # Constructing surrogate data for denoting infeasible experiment
    non_results = [len(maint_plan_list), total_time] + ["N/A"]*(len(headers) - 3)
    # Returning surrogate data
    return non_results

  # Initializing list for storing data needed to calculate expected LCOE for each feasible maintenance plan
  LCOE_data_list = []

  # For each feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Storing data needed to calculate expected LCOE for given maintenance plan
    LCOE_data_list.append((maint_plan_list[soln], plan_duration_list[soln], plan_cost_list[soln]))

  print("Now calculating uptime for all feasible solutions.")
  # Recording start time of uptime and LCOE calculations for feasible maintenance plans
  uptime_start_time = time.time()

  # Setting up parallelization process to allow uptime and LCOE to be calculate for multiple maintenance plans simultaneously
  if __name__ == "__main__":
      with multiprocessing.Pool() as pool:
          results = pool.map(expected_uptime, LCOE_data_list)

  # Calculating and reporting total uptime/LCOE calculation time
  uptime_end_time = time.time()
  print(f"Uptimes calculated in {uptime_end_time - uptime_start_time} seconds.")
  # Initializing lists for storing uptime, total cost, energy produced, and LCOE data for each feasible maintenance plan
  plan_uptime_list = []
  plan_TotalCost_list = []
  plan_EnergyProd_list = []
  plan_LCOE_list = []

  # For each feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Saving expected uptime, total cost, energy produced, and LCOE
    plan_uptime_list.append(results[soln][0])
    plan_TotalCost_list.append(results[soln][1])
    plan_EnergyProd_list.append(results[soln][2])
    plan_LCOE_list.append(results[soln][3])

  # Finding index of feasible maintenance plans with lowest expected LCOE, total cost, and energy produced
  min_LCOE_index = plan_LCOE_list.index(min(plan_LCOE_list))
  min_cost_index = plan_TotalCost_list.index(min(plan_TotalCost_list))
  max_energy_prod_index = plan_EnergyProd_list.index(max(plan_EnergyProd_list))
  # Calculating and reporting total time spent performing optimization experiment
  overall_end_time = time.time()
  total_time = overall_end_time - overall_start_time
  print(f"\nTotal time to execute code: {total_time} seconds.")
  # Returning outputs
  return [len(maint_plan_list), total_time,
          maint_plan_list[min_LCOE_index], plan_duration_list[min_LCOE_index], plan_cost_list[min_LCOE_index], c_labor*plan_duration_list[min_LCOE_index], c_downtime*(W - plan_uptime_list[min_LCOE_index]), plan_TotalCost_list[min_LCOE_index], plan_subsys_reliability_list[min_LCOE_index], plan_reliability_list[min_LCOE_index], plan_uptime_list[min_LCOE_index], plan_EnergyProd_list[min_LCOE_index], plan_LCOE_list[min_LCOE_index],
          maint_plan_list[min_cost_index], plan_duration_list[min_cost_index], plan_cost_list[min_cost_index], c_labor*plan_duration_list[min_cost_index], c_downtime*(W - plan_uptime_list[min_cost_index]), plan_TotalCost_list[min_cost_index], plan_subsys_reliability_list[min_cost_index], plan_reliability_list[min_cost_index], plan_uptime_list[min_cost_index], plan_EnergyProd_list[min_cost_index], plan_LCOE_list[min_cost_index],
          maint_plan_list[max_energy_prod_index], plan_duration_list[max_energy_prod_index], plan_cost_list[max_energy_prod_index], c_labor*plan_duration_list[max_energy_prod_index], c_downtime*(W - plan_uptime_list[max_energy_prod_index]), plan_TotalCost_list[max_energy_prod_index], plan_subsys_reliability_list[max_energy_prod_index], plan_reliability_list[max_energy_prod_index], plan_uptime_list[max_energy_prod_index], plan_EnergyProd_list[max_energy_prod_index], plan_LCOE_list[max_energy_prod_index]]

## *Sensitivity Analysis*
NOTE: Commenting has been provided for the first sensitivity analysis (Downtime Cost), but not for the rest, as they are identical, save for the parameter being varied and values being used.

### Headers

In [ ]:
# Listing headers to be used for storing data from experimental testing
headers = ["Parameter", "Number of Feasible Maintenance Plans", "CPU Time",
           "Min_LCOE Plan", "Min_LCOE Duration", "Min_LCOE Action Cost", "Min_LCOE Labor Cost", "Min_LCOE Downtime Cost", "Min_LCOE Total Cost", "Min_LCOE Subsys Reliabilities", "Min_LCOE Reliability", "Min_LCOE Uptime", "Min_LCOE Energy", "Min_LCOE LCOE",
           "Min_Cost Plan", "Min_Cost Duration", "Min_Cost Action Cost", "Min_Cost Labor Cost", "Min_Cost Downtime Cost", "Min_Cost Total Cost", "Min_Cost Subsys Reliabilities", "Min_Cost Reliability", "Min_Cost Uptime", "Min_Cost Energy", "Min_Cost LCOE",
           "Max_Energy Plan", "Max_Energy Duration", "Max_Energy Action Cost", "Max_Energy Labor Cost", "Max_Energy Downtime Cost", "Max_Energy Total Cost", "Max_Energy Subsys Reliabilities", "Max_Energy Reliability", "Max_Energy Uptime", "Max_Energy Energy", "Max_Energy LCOE"]

### Downtime Cost

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(30, 41, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value
  Outcome_tcr = deterministic_variant_tcr()
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Break Duration

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(10, 121, 10))
Results_tcr = []
headers[0] = "Break Duration (T_0)"

for l in range(len(Parameter_value_list)):
  T0 = Parameter_value_list[l]
  print(f"Testing with T_0 = {T0}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([T0] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_BreakDuration_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Maintenance Budget

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(100, 901, 100))
Results_tcr = []
headers[0] = "Maintenance Budget (C_0)"

for l in range(len(Parameter_value_list)):
  C0 = Parameter_value_list[l]
  print(f"Testing with C_0 = {C0}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([C0] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_MaintenanceBudget_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Minimum System Reliability

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(0, 1.01, 0.1))
Results_tcr = []
headers[0] = "Minimum Reliability (R_0)"

for l in range(len(Parameter_value_list)):
  R0 = Parameter_value_list[l]
  print(f"Testing with R_0 = {R0}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([R0] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_MinimumReliability_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Mission Length

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(10, 81, 10))
Results_tcr = []
headers[0] = "Mission Length (W)"

for l in range(len(Parameter_value_list)):
  W = Parameter_value_list[l]
  print(f"Testing with W = {W}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([W] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_MissionLength_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Labor Cost

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(1, 11, 1))
Results_tcr = []
headers[0] = "Labor Cost (c_labor)"

for l in range(len(Parameter_value_list)):
  c_labor = Parameter_value_list[l]
  print(f"Testing with c_labor = {c_labor}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([c_labor] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_LaborCost_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Probability of Performing Work Onshore

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(0, 1.01, 0.1))
Results_tcr = []
headers[0] = "Probability of Onshore Work (p)"

for l in range(len(Parameter_value_list)):
  p = Parameter_value_list[l]
  print(f"Testing with p = {p}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([p] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_OnshoreWorkProbability_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Onshore Cost Factor

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(1, 2.01, 0.1))
Results_tcr = []
headers[0] = "Onshore Work Cost Factor (k_cost)"
for l in range(len(Parameter_value_list)):
  k_cost = Parameter_value_list[l]
  print(f"Testing with k_cost = {k_cost}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([k_cost] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_OnshoreCostFactor_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

### Onshore Time Factor

In [ ]:
T0 = 120
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



Parameter_value_list = list(numpy.arange(1, 2.01, 0.1))
Results_tcr = []
headers[0] = "Onshore Work Time Factor (k_time)"

for l in range(len(Parameter_value_list)):
  k_time = Parameter_value_list[l]
  print(f"Testing with k_time = {k_time}...")
  Outcome_tcr = deterministic_variant_tcr()
  Results_tcr.append([k_time] + Outcome_tcr)

df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
output_path = "output_OnshoreTimeFactor_Deterministic.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

# **Uncertain Variant**

## *Decision Variable List Generator*

In [ ]:
# Defining function for generating list of maintenance action decision variables (x_ijl) that are available based on system
# Inputs: 1) number of subsystems (M), 2) number of components in each subsystem (N), and 3) maintenance actions available for each component (based on status)
# Outputs: list of tuples representing subscripts for every decision variable available based on the system
def Var_Tuple_Generator(M, N, Avail_Maint_Actions):
  # Initializing list for storing decision variable subscript tuples
  Var_Tuple_List = []

  # For each subsystem
  for i in range(M):

    # For each component in subsystem i
    for j in range(N[i]):

      # For each maintenance action available for component E_ij
      for l in range(len(Avail_Maint_Actions[i][j])):
        # Generating and store a tuple representing decision variable x_ijl
        DV_tuple = (i, j, Avail_Maint_Actions[i][j][l])
        Var_Tuple_List.append(DV_tuple)

  # Returning outputs
  return Var_Tuple_List

## *Robust Optimization Subproblem Solver Function*

In [ ]:
# Defining function for maximizing uncertainty impacts for given maintenance plan
# Inputs: as a single list, 1) list of all decision variables available for given system, 2) baseline probability of onshore work occurring for each maintenance action (p_bar),
# 3) uncertainty budget (rho), 4) duration of maintenance action l on E_ij if performed offshore (t_bar), 5) onshore time factor (k_t), 6) onshore cost factor (k_c),
# 7) selected maintenance plan as a list of binary values (x_ijl = 1 if selected, 0 otherwise), 8) expected plan duration according to baseline probabilities, and
# 9) expected plan cost according to baseline probabilities
# Outputs: 1) increase to expected plan duration from solving robust optimization subproblem (Z),
# 2) increase to expected plan cost (repairs + labour) from solving robust optimization subproblem, and
# 3) positive deviation from baseline onshore work probability for each maintenance action (d_ijl^+)
def RO_subprob(model_data):
  # Extracting data from input data list
  DV_list = model_data[0]
  p = model_data[1]
  rho = model_data[2]
  t_bar = model_data[3]
  k_t = model_data[4]
  k_c = model_data[5]
  x = model_data[6]
  nominal_duration = model_data[7]
  nominal_cost = model_data[8]
  # Initializing variable tracking total possible deviation of maintenance plan theta
  total_available_dev = 0
  # Initializing list tracking positive deviation assigned to each subsystem's actions
  subsys_uncertainties = [0]*M

  # For each maintenance action available in the problem (x_ijl)
  for (i, j, l) in DV_list:

    # If the maintenance action has a non-zero duration
    if t_bar[i][j][l] > 0:
      # Adding the maximum possible deviation from the maintenance action's p_bar to the total possible deviation of maintenance plan theta IF the maintenance action is chosen
      # in the plan (i.e., x_ijl = 1)
      total_available_dev += (1 - p[i][j][l])*x[i][j][l]

  # If the total possible deviation of maintenance plan theta does not exceed the uncertainty budget (rho)
  if total_available_dev <= rho:
    # Initializing variables for tracking total duration and repair cost for selected maintenance plan
    Total_Duration = 0
    Total_Action_Cost = 0
    # Initializing list of positive deviations from baseline onshore work probabilities (d_ijl^+)
    d_plus = {(i, j, l): 0.0 for (i, j, l) in DV_List}

    # For each maintenance action available in the problem (x_ijl)
    for (i, j, l) in DV_List:
      # Adding ONSHORE maintenance action duration and repair cost to total plan duration and repair costs if the maintenance action is selected (i.e., x_ijl = 1)
      # NOTE: If the total possible deviation of the maintenance plan does not exceed the uncertainty budget,
      # then all actions selected will have their onshore work probability set to 1
      Total_Duration += t_bar[i][j][l] * k_t * x[i][j][l]
      Total_Action_Cost += c_bar[i][j][l] * k_c * x[i][j][l]

      # If the maintenancea ction has a non-zero duration
      if t_bar[i][j][l] > 0:
        # Adding maintenance action's maximium possible deviation from p_bar to corresponding subsystem's allocated uncertainty if maintenance action is selected (i.e., x_ijl = 1)
        subsys_uncertainties[i] += (1 - p[i][j][l])*x[i][j][l]

    # Calculating increase to maintenance plan duration and action and labour costs caused by robust optimization
    RO_Duration_Increase = Total_Duration - nominal_duration
    RO_Cost_Increase = Total_Action_Cost + Total_Duration*c_labor - (nominal_cost + c_labor*nominal_duration)
    # Returning outputs
    return RO_Duration_Increase, RO_Cost_Increase, subsys_uncertainties

  # Setting up environment for performing optimization with Gurobi (NOTE: this is required if using multiprocessing to simultaneously optimize for multiple plans)
  env = Env(empty=True)
  for k, v in params.items():
    env.setParam(k, v)
  env.setParam("Threads", 1)
  env.setParam("LogToConsole", 0)
  env.start()

  # Initializing Gurobi model for optimization
  model = Model(env = env)

  # Introducing dual variables corresponding to positive deviation constraints in robust optimization subproblem (delta_ijl^+)
  delta_plus = model.addVars(DV_List, vtype = GRB.CONTINUOUS, lb = 0, name = "delta_plus")
  # Introducing dual variable corresponding to uncertainty budget constraint (nu)
  nu = model.addVar(vtype = GRB.CONTINUOUS, lb = 0, name = "nu")

  # Declaring objective function for dualized robust optimization subproblem
  # (increase in maintenance plan duration incurred from increasing positive deviations of onshore work probabilities)
  model.setObjective(quicksum((1 - p[i][j][l]) * delta_plus[i, j, l] for (i, j, l) in DV_List) + rho * nu, GRB.MINIMIZE)

  # Declaring dualized robust optimization subproblem constraints
  pos_dev_constrs = {}
  for (i, j, l) in DV_List:
      pos_dev_constrs[i, j, l] = model.addConstr(
          delta_plus[i, j, l] + nu >= t_bar[i][j][l] * (k_t - 1) * x[i][j][l],
          name=str(f"pos_dev_constr_{i}_{j}_{l}")
      )

  # Solving problem
  model.optimize()
  # Recording increase to maintenance plan duration caused by robust optimization
  RO_Duration_Increase = model.ObjVal
  # Extracting dual variables of solved problem corresponding to positive deviations from baseline onshore work probabilities (d_ijl^+)
  d_plus = {(i, j, l): pos_dev_constrs[i, j, l].Pi for (i, j, l) in DV_List}
  # Initializing variable for tracking total repair cost of maintenance plan
  Total_Action_Cost = 0

  # For each maintenance action available in the problem (x_ijl)
  for (i, j, l) in DV_List:
    # Adding dual variable value to total positive deviation allocated to corresponding subsystem
    subsys_uncertainties[i] += d_plus[i, j, l]
    # Adding expected repair cost of maintenance action according to baseline onshore work probability and positive deviation
    Total_Action_Cost += c_bar[i][j][l] * x[i][j][l] * (1 + (k_c - 1) * (p[i][j][l] + d_plus[i, j, l]))

  # Calculating increase to maintenance plan's repair and action costs caused by robust optimization
  RO_Cost_Increase = Total_Action_Cost + RO_Duration_Increase*c_labor - nominal_cost
  # Returning outputs
  return RO_Duration_Increase, RO_Cost_Increase, subsys_uncertainties

## *Binary Plan Generator Function*

In [ ]:
# Defining function for converting maintenance plans under consideration to lists of binary values
# Inputs: 1) number of subsystems (M), 2) number of components in each subsystem (N), 3) number of maintenance actions available for each component (L), 4) list of maintenance plans under consideration
# Outputs: list of lists of binary values representing maintenance plans under consideration
def binary_plan_generator(M, N, L, solutions):
  # Initializing list for storing binary maintenance plans
  binary_plans = []

  # For each maintenance plan under consideration
  for n in range(len(solutions)):
    # Initializing list of binary decision variable values for maintenance plan n
    x = []

    # For each subsystem
    for i in range(M):
      # Initializing list of binary decision variable values for subsystem i maintenance plan
      subsys_x = []

      # For each component in subsystem i
      for j in range(N[i]):
        # Initializing list of binary decision variable values for component E_ij maintenance plan
        comp_x = []

        # For each maintenance action available
        for l in range(L):

          # If maintenance action l is selected for component E_ij in maintenance plan n
          if solutions[n][i][j] == l:
            # Storing 1 value corresponding to x_ijl
            comp_x.append(1)
          # Else
          else:
            # Storing 0 value corresponding to x_ijl
            comp_x.append(0)

        # Appending component E_ij binary maintenance plan to subsystem list
        subsys_x.append(comp_x)

      # Appending subsystem i binary maintenance plan to complete maintenance plan list
      x.append(subsys_x)

    # Storing complete binary plan for maintenance plan n
    binary_plans.append(x)

  # Returning outputs
  return binary_plans

## *Uptime and LCOE Calculator*

In [ ]:
# Defining the function for calculating the expected uptime of the asset during a mission after executing a specific maintenance plan
# Inputs: relevant data for maintenance plan under consideration (1) maintenance plan under consideration, 2) expected time to perform maintenance plan, 3) expected repair cost to perform maintenance plan,
# 4) increase in repair and labour costs from robust optimization)
# Outputs: expected 1) uptime, 2) total cost of the selected plan, 3) energy production, 4) LCOE
def expected_uptime(Maint_Plan_Params):
  # Loading relevant data for selected maintenance plan (actions selected, total duration, total repair cost, increase in repair and labour costs from robust optimization)
  maint_plan = Maint_Plan_Params[0]
  Maint_Time = Maint_Plan_Params[1]
  Maint_Cost = Maint_Plan_Params[2]
  RO_Cost_Increase = Maint_Plan_Params[3]
  # Calculating expected uptime of the asset according to the mission length and selected maintenance plan (U_theta(W))
  avg_uptime, throwaway = scipy.integrate.quad(rel_calc, 0, W, maint_plan)
  # Calculating the expected total cost of the selected maintenance plan (repair, labour, downtime, and increase from robust optimization uncertainty)
  Total_Cost = Maint_Cost + c_labor*Maint_Time + c_downtime*(W - avg_uptime) + RO_Cost_Increase
  # Calculating the expected energy produced by the asset according to the expected uptime
  Energy_Prod = energy_rate*avg_uptime

  # If energy is produced (i.e., the device is functional when the mission begins)
  if Energy_Prod > 0:
    # Calculating expected LCOE of the asset according to the selected maintenance plan (zeta_theta)
    LCOE = Total_Cost/Energy_Prod
  # Else
  else:
    # Setting expected LCOE of the asset equal to infinity (division by zero)
    LCOE = numpy.inf

  # Returning outputs
  return [avg_uptime, Total_Cost, Energy_Prod, LCOE]

## *Additional Parameters Needed for Uncertain Variant*

In [ ]:
# Uncertainty budget (rho)
rho = 6

# Onshore maintenance baseline probabilities (p_bar^cm and p_bar^pm)
# Same structure as T_site and C_site
p_nominal = [[[[0.00, 0.30, 0.40, 0.60],[0.00, 0.00, 0.20, 0.30]], [[0.00, 0.30, 0.40, 0.60],[0.00, 0.00, 0.20, 0.30]], [[0.00, 0.30, 0.40, 0.60],[0.00, 0.00, 0.20, 0.30]], [[0.00, 0.30, 0.40, 0.60],[0.00, 0.00, 0.20, 0.30]]],
             [[[0.00, 0.40, 0.53, 0.80],[0.00, 0.00, 0.27, 0.40]], [[0.00, 0.40, 0.53, 0.80],[0.00, 0.00, 0.27, 0.40]]],
             [[[0.00, 0.35, 0.47, 0.70],[0.00, 0.00, 0.23, 0.35]], [[0.00, 0.35, 0.47, 0.70],[0.00, 0.00, 0.23, 0.35]], [[0.00, 0.35, 0.47, 0.70],[0.00, 0.00, 0.23, 0.35]]],
             [[[0.00, 0.45, 0.60, 0.90],[0.00, 0.00, 0.30, 0.45]], [[0.00, 0.45, 0.60, 0.90],[0.00, 0.00, 0.30, 0.45]]],
             [[[0.00, 0.25, 0.33, 0.50],[0.00, 0.00, 0.17, 0.25]], [[0.00, 0.25, 0.33, 0.50],[0.00, 0.00, 0.17, 0.25]], [[0.00, 0.25, 0.33, 0.50],[0.00, 0.00, 0.17, 0.25]]]]



# Creating copy of offshore maintenance cost and duration data
# NOTE: Due to changes in how baseline onshore probabilities are modelled, C_expected and T_expected must be re-calculated for the uncertain variant.
C_expected = copy.deepcopy(C_site)
T_expected = copy.deepcopy(T_site)
# For each subsystem
for i in range(M):
  # For each component in subsystem i
  for j in range(N[i]):
    # For both lists storing corrective and preventive maintenance data
    for k in range(2):
      # For each maintenance level
      for l in range(L):
        # Calculating expected cost and duration of each maintenance action (C_ijl and T_ijl) based on onshore cost and time factors (k_c and k_t) and baseline probability of onshore work (p_bar_ijl)
        C_expected[i][j][k][l] = C_expected[i][j][k][l]*(k_cost*p_nominal[i][j][k][l] + 1 - p_nominal[i][j][k][l])
        T_expected[i][j][k][l] = T_expected[i][j][k][l]*(k_time*p_nominal[i][j][k][l] + 1 - p_nominal[i][j][k][l])



# NOTE: The following code simplifies the duration, cost, and probability parameter lists by removing the dimension related to CM/PM
# (i.e., we only consider CM actions for failed components and PM actions for working components)
# Maintenance action duration for offshore work (t_bar)
t_bar = []
for i in range(M):
  subsys_t_bar = []
  for j in range(N[i]):
    comp_t_bar = T_site[i][j][Component_Status[i][j]]
    subsys_t_bar.append(comp_t_bar)
  t_bar.append(subsys_t_bar)



# Maintenance repair cost for offshore work (c_bar)
c_bar = []
for i in range(M):
  subsys_c_bar = []
  for j in range(N[i]):
    comp_c_bar = C_site[i][j][Component_Status[i][j]]
    subsys_c_bar.append(comp_c_bar)
  c_bar.append(subsys_c_bar)



# Baseline probability of onshore work for maintenance action (p_bar)
p_bar = []
for i in range(M):
  subsys_p_bar = []
  for j in range(N[i]):
    comp_p_bar = p_nominal[i][j][Component_Status[i][j]]
    subsys_p_bar.append(comp_p_bar)
  p_bar.append(subsys_p_bar)



# Available maintenance actions for each subsystem's components
Avail_Maint_Actions = []
for i in range(M):
  Avail_Maint = avail_maint_list(i)
  Avail_Maint_Actions.append(Avail_Maint)



# List of decision variables available based on system to be repaired (x_ijl)
DV_List = Var_Tuple_Generator(M, N, Avail_Maint_Actions)

## *Uncertain MRE SMP Solution Methodology Master Function*

In [ ]:
# Defining master function for finding optimal maintenance plan
# Inputs: none
# Outputs: 1) number of feasible solutions, 2) computation time,
# For LCOE minimization: 3) optimal maintenance plan and associated 4) minimum expected duration, 5) increase in duration due to RO, 6) total duration (minimum expected + RO increase), 7) repair cost, 8) labour cost, 9) downtime cost, 10) increase in cost due to RO, 11) total cost, 12) uncertainty allocated to each subsystem, 13) subsystem conditional reliabilities, 14) system conditional reliability, 15) expected uptime, 16) expected energy produced, and 17) expected LCOE
# For cost minimization: 18) optimal maintenance plan and associated 19) minimum expected duration, 20) increase in duration due to RO, 21) total duration (minimum expected + RO increase), 22) repair cost, 23) labour cost, 24) downtime cost, 25) increase in cost due to RO, 26) total cost, 27) uncertainty allocated to each subsystem, 28) subsystem conditional reliabilities, 29) system conditional reliability, 30) expected uptime, 31) expected energy produced, and 32) expected LCOE
# For energy production maximization: 33) optimal maintenance plan and associated 34) minimum expected duration, 35) increase in duration due to RO, 36) total duration (minimum expected + RO increase), 37) repair cost, 38) labour cost, 39) downtime cost, 40) increase in cost due to RO, 41) total cost, 42) uncertainty allocated to each subsystem, 43) subsystem conditional reliabilities, 44) system conditional reliability, 45) expected uptime, 46) expected energy produced, and 47) expected LCOE
def uncertain_variant_tcr():
  # Recording start time of experiment
  overall_start_time = time.time()
  print(f"Generating plans with time, cost, and reliability constraints (T_0 = {T0}, C_0 = {C0}, R_0 = {R0})...")
  # Recording start time of feasible plan generation
  soln_generation_start_time = time.time()
  # Generating all complete, initially feasible maintenance plans based on given parameters
  maint_plan_list, plan_duration_list, plan_cost_list, plan_reliability_list, plan_subsys_reliability_list = plan_list_generator_time_cost_reliability_constraints()
  # Calculating and reporting time spent generating maintenance plans
  soln_generation_end_time = time.time()
  print(f"{len(maint_plan_list)} plans generated!")
  print(f"Solution generation took {soln_generation_end_time - soln_generation_start_time} seconds.")

  # If no feasible maintenance plans were constructed
  if len(maint_plan_list) == 0:
    # Calculating total experiment time and reporting infeasibility
    overall_end_time = time.time()
    total_time = overall_end_time - overall_start_time
    print("No feasible solutions found.")
    # Constructing surrogate data for denoting infeasible experiment
    non_results = [len(maint_plan_list), total_time] + ["N/A"]*(len(headers) - 3)
    # Returning surrogate data
    return non_results

  # Initializing lists for storing data related to RO subproblem solving for the complete, initially feasible maintenance plans
  plan_subsys_uncertainty_list = []
  plan_RO_duration_increase_list = []
  plan_RO_cost_increase_list = []
  # Creating binary equivalent of each initially feasible maintenance plan
  binary_plans = binary_plan_generator(M, N, L, maint_plan_list)
  # Initializing list for storing data needed to perform RO on initially feasible maintenance plans
  model_data_list = []

  # For each initially feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Storing relevant data for performing RO
    model_data_list.append((DV_List, p_bar, rho, t_bar, k_time, k_cost, binary_plans[soln], plan_duration_list[soln], plan_cost_list[soln]))

  print("Now performing robust optimization for all feasible solutions.")
  # Recording start time of RO process for all initially feasible maintenance plans
  RO_start_time = time.time()

  # Setting up parallelization process to allow RO subproblem to be solved for multiple maintenance plans simultaneously
  if __name__ == "__main__":
    with multiprocessing.Pool(processes = (multiprocessing.cpu_count() - 1)) as pool:
        RO_model_results = pool.map(RO_subprob, model_data_list)

  # Calculating and reporting time to perform RO on all initially feasible maintenance plans
  RO_end_time = time.time()
  print(f"RO performed in {RO_end_time - RO_start_time} seconds.")

  # For each initially feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Storing associated increase in duration and cost from solving RO subproblem, as well as list reporting how uncertainty was allocated to each subsystem's repairs
    plan_RO_duration_increase_list.append(RO_model_results[soln][0])
    plan_RO_cost_increase_list.append(RO_model_results[soln][1])
    plan_subsys_uncertainty_list.append(RO_model_results[soln][2])

  print("Now culling solutions based on RO results.")
  # Initializing lists for storing complete maintenance plans that remain feasible after RO is applied (and their associated data)
  new_maint_plan_list = []
  new_plan_duration_list = []
  new_plan_cost_list = []
  new_plan_reliability_list = []
  new_plan_subsys_reliability_list = []
  new_binary_plans = []
  new_plan_RO_duration_increase_list = []
  new_plan_RO_cost_increase_list = []
  new_plan_subsys_uncertainty_list = []
  # Initializing list for storing data needed to calculate expected LCOE for each feasible maintenance plan
  LCOE_data_list = []

  # For each initially feasible maintenance plan
  for n in range(len(maint_plan_list)):

    # If the maintenance plan's total time and cost (after applying RO) still do not violate break duraiton and budget constraints
    if (plan_duration_list[n] + plan_RO_duration_increase_list[n]) <= T0 and (plan_cost_list[n] + plan_duration_list[n]*c_labor + plan_RO_cost_increase_list[n]) <= C0:
      # Saving maintenance plan and associated data
      new_maint_plan_list.append(maint_plan_list[n])
      new_plan_duration_list.append(plan_duration_list[n])
      new_plan_cost_list.append(plan_cost_list[n])
      new_plan_reliability_list.append(plan_reliability_list[n])
      new_plan_subsys_reliability_list.append(plan_subsys_reliability_list[n])
      new_binary_plans.append(binary_plans[n])
      new_plan_RO_duration_increase_list.append(plan_RO_duration_increase_list[n])
      new_plan_RO_cost_increase_list.append(plan_RO_cost_increase_list[n])
      new_plan_subsys_uncertainty_list.append(plan_subsys_uncertainty_list[n])
      # Storing data needed to calculate expected LCOE for given maintenance plan
      LCOE_data_list.append((maint_plan_list[n], plan_duration_list[n], plan_cost_list[n], plan_RO_cost_increase_list[n]))

  # Replacing lists of feasible maintenance plans and their associated data with culled lists after RO
  maint_plan_list = new_maint_plan_list
  plan_duration_list = new_plan_duration_list
  plan_cost_list = new_plan_cost_list
  plan_reliability_list = new_plan_reliability_list
  plan_subsys_reliability_list = new_plan_subsys_reliability_list
  binary_plans = new_binary_plans
  plan_RO_duration_increase_list = new_plan_RO_duration_increase_list
  plan_RO_cost_increase_list = new_plan_RO_cost_increase_list
  plan_subsys_uncertainty_list = new_plan_subsys_uncertainty_list

  # If no maintenance plans have remained feasible after performing RO
  if len(maint_plan_list) == 0:
    # Calculating total experiment time and reporting infeasibility
    overall_end_time = time.time()
    total_time = overall_end_time - overall_start_time
    print("No feasible solutions found.")
    # Constructing surrogate data for denoting infeasible experiment
    non_results = [len(maint_plan_list), total_time] + ["N/A"]*(len(headers) - 3)
    # Returning surrogate data
    return non_results

  print("Now calculating uptime for all feasible solutions.")
  # Recording start time of uptime and LCOE calculations for remaining feasible maintenance plans
  uptime_start_time = time.time()

  # Setting up parallelization process to allow uptime and LCOE to be calculate for multiple maintenance plans simultaneously
  if __name__ == "__main__":
      with multiprocessing.Pool(processes = (multiprocessing.cpu_count() - 1)) as pool:
          results = pool.map(expected_uptime, LCOE_data_list)

  # Calculating and reporting total uptime/LCOE calculation time
  uptime_end_time = time.time()
  print(f"Uptimes calculated in {uptime_end_time - uptime_start_time} seconds.")
  # Initializing lists for storing uptime, total cost, energy produced, and LCOE data for each feasible maintenance plan
  plan_uptime_list = []
  plan_TotalCost_list = []
  plan_EnergyProd_list = []
  plan_LCOE_list = []

  # For each feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Saving expected uptime, total cost, energy produced, and LCOE
    plan_uptime_list.append(results[soln][0])
    plan_TotalCost_list.append(results[soln][1])
    plan_EnergyProd_list.append(results[soln][2])
    plan_LCOE_list.append(results[soln][3])

  # Finding index of feasible maintenance plans with lowest expected LCOE, total cost, and energy produced
  min_LCOE_index = plan_LCOE_list.index(min(plan_LCOE_list))
  min_cost_index = plan_TotalCost_list.index(min(plan_TotalCost_list))
  max_energy_prod_index = plan_EnergyProd_list.index(max(plan_EnergyProd_list))
  # Calculating and reporting total time spent performing optimization experiment
  overall_end_time = time.time()
  total_time = overall_end_time - overall_start_time
  print(f"Final number of solutions: {len(maint_plan_list)}")
  print(f"Total time to execute code: {total_time} seconds.")
  # Returning outputs
  return [len(maint_plan_list), total_time,
          maint_plan_list[min_LCOE_index], plan_duration_list[min_LCOE_index], plan_RO_duration_increase_list[min_LCOE_index], plan_duration_list[min_LCOE_index] + plan_RO_duration_increase_list[min_LCOE_index], plan_cost_list[min_LCOE_index], c_labor*plan_duration_list[min_LCOE_index], c_downtime*(W - plan_uptime_list[min_LCOE_index]), plan_RO_cost_increase_list[min_LCOE_index], plan_TotalCost_list[min_LCOE_index], plan_subsys_uncertainty_list[min_LCOE_index], plan_subsys_reliability_list[min_LCOE_index], plan_reliability_list[min_LCOE_index], plan_uptime_list[min_LCOE_index], plan_EnergyProd_list[min_LCOE_index], plan_LCOE_list[min_LCOE_index],
          maint_plan_list[min_cost_index], plan_duration_list[min_cost_index], plan_RO_duration_increase_list[min_cost_index], plan_duration_list[min_cost_index] + plan_RO_duration_increase_list[min_cost_index], plan_cost_list[min_cost_index], c_labor*plan_duration_list[min_cost_index], c_downtime*(W - plan_uptime_list[min_cost_index]), plan_RO_cost_increase_list[min_cost_index], plan_TotalCost_list[min_cost_index], plan_subsys_uncertainty_list[min_cost_index], plan_subsys_reliability_list[min_cost_index], plan_reliability_list[min_cost_index], plan_uptime_list[min_cost_index], plan_EnergyProd_list[min_cost_index], plan_LCOE_list[min_cost_index],
          maint_plan_list[max_energy_prod_index], plan_duration_list[max_energy_prod_index], plan_RO_duration_increase_list[max_energy_prod_index], plan_duration_list[max_energy_prod_index] + plan_RO_duration_increase_list[max_energy_prod_index], plan_cost_list[max_energy_prod_index], c_labor*plan_duration_list[max_energy_prod_index], c_downtime*(W - plan_uptime_list[max_energy_prod_index]), plan_RO_cost_increase_list[max_energy_prod_index], plan_TotalCost_list[max_energy_prod_index], plan_subsys_uncertainty_list[max_energy_prod_index], plan_subsys_reliability_list[max_energy_prod_index], plan_reliability_list[max_energy_prod_index], plan_uptime_list[max_energy_prod_index], plan_EnergyProd_list[max_energy_prod_index], plan_LCOE_list[max_energy_prod_index]]

## *Sensitivity Analysis*

In [ ]:
# Listing headers to be used for storing data from experimental testing
headers = ["Parameter", "Number of Feasible Maintenance Plans", "CPU Time",
           "Min_LCOE Plan", "Min_LCOE Nominal Duration", "Min_LCOE RO Duration Increase", "Min_LCOE Total Duration", "Min_LCOE Action Cost", "Min_LCOE Labor Cost", "Min_LCOE Downtime Cost", "Min_LCOE RO Cost Increase", "Min_LCOE Total Cost", "Min_LCOE RO Subsys Uncertainties", "Min_LCOE Subsys Reliabilities", "Min_LCOE Reliability", "Min_LCOE Uptime", "Min_LCOE Energy", "Min_LCOE LCOE",
           "Min_Cost Plan", "Min_Cost Nominal Duration", "Min_Cost RO Duration Increase", "Min_Cost Total Duration", "Min_Cost Action Cost", "Min_Cost Labor Cost", "Min_Cost Downtime Cost", "Min_Cost RO Cost Increase", "Min_Cost Total Cost", "Min_Cost RO Subsys Uncertainties", "Min_Cost Subsys Reliabilities", "Min_Cost Reliability", "Min_Cost Uptime", "Min_Cost Energy", "Min_Cost LCOE",
           "Max_Energy Plan", "Max_Energy Nominal Duration", "Max_Energy RO Duration Increase", "Max_Energy Total Duration", "Max_Energy Action Cost", "Max_Energy Labor Cost", "Max_Energy Downtime Cost", "Max_Energy RO Cost Increase", "Max_Energy Total Cost", "Max_Energy RO Subsys Uncertainties", "Max_Energy Subsys Reliabilities", "Max_Energy Reliability", "Max_Energy Uptime", "Max_Energy Energy", "Max_Energy LCOE"]

### Uncertainty Budget

In [ ]:
# Listing default parameter values
T0 = 50
C0 = 500
R0 = 0.5
W = 60
c_labor = 5
c_downtime = 33
k_cost = 1.25
k_time = 1.5
rho = 6



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(0, 6.1, 0.25))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Uncertainty Budget (rho)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  rho = Parameter_value_list[l]
  print(f"\nTesting with rho = {rho}...")
  # Performing optimization experiment with changed parameter value
  Outcome_tcr = uncertain_variant_tcr()
   # Storing solution results from experiment
  Results_tcr.append([rho] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_UncertaintyBudget_Uncertain_50T0_V2.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

## *Out-of-Sample Plan Assessment*

### Subfunctions

In [ ]:
# Defining function for calculating actual expected duration of maintenance plan based on out-of-sample onshore work probabilities
# Inputs: 1) out-of-sample onshore work probabilities from trial and 2) selected maintenance plan (in binary format)
# Outputs: 1) actual expected plan duration and 2) expected overtime based on trial
def Trial_Plan_Duration_Calculator(trial_p, maint_plan):
  # Initializing variables for tracking actual expected plan duration and overtime
  Plan_Duration = 0
  Overtime = 0
  # For each maintenance action available in the problem (x_ijl)
  for (i, j, l) in DV_List:
    # Adding actual expected duration of maintenance action based on trial onshore work probability (p_ijl)
    Plan_Duration += t_bar[i][j][l] * maint_plan[i][j][l] * (1 + (k_time - 1) * trial_p[i+1][j+1][l])
  # If total expected plan duration exceeds break duration
  if Plan_Duration > T0:
    # Calculting expected overtime of maintenance plan based on trial
    Overtime = Plan_Duration - T0
  # Returning outputs
  return Plan_Duration, Overtime

# Defining function for calculating actual expected cost of maintenance plan based on out-of-sample onshore work probabilities
# Inputs: 1) cost rate of overtime, 2, out-of-sample onshore work probabilities from trial, 3) actual expected plan duration, 4) expected uptime of system, 5) selected maintenance plan (in binary format), and
# 6) expected overtime
# Outputs: 1) expected plan cost of repairs/labour/overtime, 2) expected overtime cost, 3) expected amount of exceedance from set maintenance budget, and 4) expected total plan cost
def Trial_Plan_Cost_Calculator(c_overtime, trial_p, Plan_Duration, Uptime, maint_plan, Overtime):
  # Initializing variables for tracking total amount maintenance budget is exceeded by, total cost of overtime, and total expected plan cost
  Budget_Exceedance = 0
  Overtime_Cost = Overtime * c_overtime
  Plan_Cost = c_labor * Plan_Duration + Overtime_Cost
  # For each maintenance action available in the problem (x_ijl)
  for (i, j, l) in DV_List:
    # Adding actual expected cost of maintenance action based on trial onshore work probability (p_ijl)
    Plan_Cost += c_bar[i][j][l] * maint_plan[i][j][l] * (1 + (k_cost - 1) * trial_p[i+1][j+1][l])
  # If total expected plan cost exceeds budget
  if Plan_Cost > C0:
    # Calculating expected amount maintenance budget is exceeded by
    Budget_Exceedance = Plan_Cost - C0
  # Calculating total expected plan cost (including downtime)
  Total_Plan_Cost = Plan_Cost + c_downtime * (W - Uptime)
  # Returning outputs
  return Plan_Cost, Overtime_Cost, Budget_Exceedance, Total_Plan_Cost

# Defining function for performing complete evaluation of maintenance plan performance under out-of-sample trials
# Inputs: 1) cost rate of overtime (c_o), 2) selected maintenance plan in binary format, 3) expected uptime of maintenance plan (U_theta(W)), 4) expected energy produced from maintenance plan (EP(W)),
# 5) uncertainty budget (rho), and 6) empty dataframe formatted for recording duration, cost, and LCOE of maintenance plan under each trial
# Outputs: 1) number of trials tested, 2) number of trials that resulted in overtime, 3) proportion of trials that resulted in overtime, 4) number of trials that resulted in budget exceedance,
# 5) proportion of trials that resulted in budget exceedance, 6) average duration of maintenance plan, 7) standard deviation of maintenance plan duration,
# 8) average amount of slack in break duration constraint for maintenance plan, 9) average amount of overtime incurred for maintenance plan across all trials,
# 10) standard deviation of maintenance plan overtime across all trials, 11) average amount of overtime incurred for maintenance plan across trials that had overtime,
# 12) standard deviation of maintenance plan overtime across trials that had overtime, 13) average maintenance plan cost (excluding downtime), 14) standard deviation of maintenance plan cost (excluding downtime)
# 15) average amount of slack in maintenance budget constraint for maintenance plan, 16) average total maintenance plan cost, 17) standard deviation of total maintenance plan cost,
# 18) average overtime cost of maintenance plan across all trials, 19) standard deviation of maintenance plan overtime cost across all trials,
# 20) average overtime cost of maintenance plan across trials that had overtime, 21) standard deviation of maintenance plan overtime cost across trials that had overtime,
# 22) average budget exceedance for maintenance plan across all trials, 23) standard deviation of maintenance plan budget exceedance across all trials,
# 24) average budget exceedance for maintenance plan across trials that had budget exceedance, 25) standard deviation of maintenance plan budget exceedance across trials that had budget exceedance
# 26) average LCOE of maintenance plan, 27) standard devaiation of maintenance plan LCOE
def solution_assessment_function(c_overtime, Binary_Plan, Uptime, Energy_Produced, rho, Durations_DF):
  # Accessing onshore work probability trials data
  test_data = pandas.read_csv('RO_Prob_Data_14Comp.txt', sep='\t', header=0)
  # Determining number of trials to be performed
  num_trials = test_data['Trial'].max()
  # Initializing variables for tracking number of trials that resulted in break duration or budget exceedance
  break_duration_exceeded = 0
  budget_exceeded = 0
  # Initializing lists for storing maintenance plan performance data across trials
  Trial_Plan_Durations = []
  Trial_Plan_Costs = []
  Trial_Plan_Total_Costs = []
  Trial_Plan_Overtimes = []
  Trial_Plan_Overtime_Costs = []
  Trial_Plan_Budget_Exceedances = []
  Trial_Plan_LCOEs = []
  # For each trial
  for trial in range(1, num_trials + 1):
    # Extracting trial data and mapping onshore work probabilities to container
    trial_data = test_data[test_data['Trial'] == trial]
    trial_p = defaultdict(lambda: defaultdict(dict))
    for _, row in trial_data.iterrows():
        i, j, l, p_actual = row['i'], row['j'], row['l'], row['p_actual']
        trial_p[i][j][l] = p_actual
    # Calculating and storing maintenance plan duration and overtime based on trial probabilities
    Plan_Duration, Overtime = Trial_Plan_Duration_Calculator(trial_p, Binary_Plan)
    Trial_Plan_Durations.append(Plan_Duration)
    Trial_Plan_Overtimes.append(Overtime)
    # Calculating and storing maintenance plan cost (excluding downtime), overtime cost, amount of budget exceedance, and total cost based on trial probabilities
    Plan_Cost, Overtime_Cost, Budget_Exceedance, Plan_Total_Cost = Trial_Plan_Cost_Calculator(c_overtime, trial_p, Plan_Duration, Uptime, Binary_Plan, Overtime)
    Trial_Plan_Costs.append(Plan_Cost)
    Trial_Plan_Total_Costs.append(Plan_Total_Cost)
    Trial_Plan_Overtime_Costs.append(Overtime_Cost)
    Trial_Plan_Budget_Exceedances.append(Budget_Exceedance)
    # Calculating maintenance plan LCOE based on trial probabilities
    Plan_LCOE = Plan_Total_Cost/Energy_Produced
    # Storing relevant data from trial in separate dataframe
    Durations_DF.loc[len(Durations_DF)] = [rho, Plan_Duration, Plan_Cost, Plan_LCOE]
    # Saving trial LCOE
    Trial_Plan_LCOEs.append(Plan_LCOE)
    # If maintenance plan duration exceeded break duration based on trial probabilities
    if Plan_Duration > T0:
      # Incrementing counter for tracking number of trials resulting in break duration exceedance
      break_duration_exceeded += 1
    # If maintenance plan cost (excluding downtime) exceeded budget based on trial probabilities
    if Plan_Cost > C0:
      # Incrementing counter for tracking number of trials resulting in budget exceedance
      budget_exceeded += 1
  # Calculating outputs
  break_duration_exceeded_proportion = break_duration_exceeded / num_trials
  budget_exceeded_proportion = budget_exceeded / num_trials
  avg_duration = numpy.average(Trial_Plan_Durations)
  std_dev_duration = numpy.std(Trial_Plan_Durations)
  avg_plan_duration_slack = max(T0 - avg_duration, 0)
  avg_plan_ot = numpy.average(Trial_Plan_Overtimes)
  std_dev_plan_ot = numpy.std(Trial_Plan_Overtimes)
  non_zero_ots = [v for v in Trial_Plan_Overtimes if v != 0]
  avg_ot = numpy.average(non_zero_ots)
  std_dev_ot = numpy.std(non_zero_ots)
  avg_cost = numpy.average(Trial_Plan_Costs)
  std_dev_cost = numpy.std(Trial_Plan_Costs)
  avg_plan_cost_slack = max(C0 - avg_cost, 0)
  avg_total_cost = numpy.average(Trial_Plan_Total_Costs)
  std_dev_total_cost = numpy.std(Trial_Plan_Total_Costs)
  avg_plan_ot_cost = numpy.average(Trial_Plan_Overtime_Costs)
  std_dev_plan_ot_cost = numpy.std(Trial_Plan_Overtime_Costs)
  non_zero_ot_costs = [v for v in Trial_Plan_Overtime_Costs if v != 0]
  avg_ot_cost = numpy.average(non_zero_ot_costs)
  std_dev_ot_cost = numpy.std(non_zero_ot_costs)
  avg_plan_budget_exceed = numpy.average(Trial_Plan_Budget_Exceedances)
  std_dev_plan_budget_exceed = numpy.std(Trial_Plan_Budget_Exceedances)
  non_zero_budget_exceed = [v for v in Trial_Plan_Budget_Exceedances if v != 0]
  avg_budget_exceed = numpy.average(non_zero_budget_exceed)
  std_dev_budget_exceed = numpy.std(non_zero_budget_exceed)
  avg_lcoe = numpy.average(Trial_Plan_LCOEs)
  std_dev_lcoe = numpy.std(Trial_Plan_LCOEs)
  # Returning outputs
  return num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe

### Main Assessment Code

In [ ]:
# Setting cost rate of overtime
c_overtime = 10*c_labor

# Listing headers to be used for storing data from experimental testing with trial data
results_headers = ["Uncertainty Budget (rho)", "Number of Feasible Maintenance Plans", "CPU Time", "Plan", "Nominal Duration", "RO Duration Increase", "Total Duration", "Action Cost", "Labor Cost", "Downtime Cost", "RO Cost Increase", "Total Cost", "RO Subsys Uncertainties", "Subsys Reliabilities", "Reliability", "Uptime", "Energy Produced", "LCOE", "Number of Trials", "Break Duration Exceedance Count", "Break Duration Exceedance Proportion", "Budget Exceedance Count", "Budget Exceedance Proportion", "Avg. Duration", "Duration SD", "Avg. Duration Slack", "Avg. Overtime", "Overtime SD", "Avg. Non-Zero Overtime", "Non-Zero Overtime SD", "Avg. Plan Cost", "Plan Cost SD", "Avg. Plan Cost Slack", "Avg. Total Cost", "Total Cost SD", "Avg. Overtime Cost", "Overtime Cost SD", "Avg. Non-Zero Overtime Cost", "Non-Zero Overtime Cost SD", "Avg. Budget Exceedance", "Budget Exceedance SD", "Avg. Non-Zero Budget Exceedance", "Non-Zero Budget Exceedance SD", "Avg. LCOE", "LCOE SD"]
results_durations_headers = ["rho", "Duration", "Cost", "LCOE"]

# Setting up dataframes for storing experimental test results
LCOE_plan_analysis = pandas.DataFrame(columns = results_headers)
Cost_plan_analysis = pandas.DataFrame(columns = results_headers)
Energy_plan_analysis = pandas.DataFrame(columns = results_headers)
LCOE_Durations_DF = pandas.DataFrame(columns = results_durations_headers)
Cost_Durations_DF = pandas.DataFrame(columns = results_durations_headers)
Energy_Durations_DF = pandas.DataFrame(columns = results_durations_headers)

# Defining file paths and sheet names for storing experimental test results
file_path = "Trial_Results_V2.xlsx"
file_path_durations = "Trial_Durations_V2.xlsx"
LCOE_sheet_name = "LCOE"
Cost_sheet_name = "Cost"
Energy_sheet_name = "Energy"

# Accessing optimal maintenance plan data from performing robust optimization with varied uncertainty budget
RO_results = pandas.read_excel("output_UncertaintyBudget_Uncertain_50T0_V2.xlsx")
RO_results["Min_LCOE Plan"] = RO_results["Min_LCOE Plan"].apply(ast.literal_eval)
RO_results["Min_Cost Plan"] = RO_results["Min_Cost Plan"].apply(ast.literal_eval)
RO_results["Max_Energy Plan"] = RO_results["Max_Energy Plan"].apply(ast.literal_eval)
# For each test with a different uncertainty budget
for index, row in RO_results.iterrows():
  print("Row index:", index)
  # Extracting relevant data for all maintenance plans
  rho = row['Uncertainty Budget (rho)']
  num_plans = row['Number of Feasible Maintenance Plans']
  comp_time = row['CPU Time']
  print(f"Now assessing optimal plans at rho = {rho}...")
  print(f"Assessing Min_LCOE Plan...")
  # Extracting relevant data for LCOE-minimization maintenance plan
  plan = row["Min_LCOE Plan"]
  print(f"Plan: {plan}")
  nominal_duration = row["Min_LCOE Nominal Duration"]
  ro_duration_increase = row["Min_LCOE RO Duration Increase"]
  total_duration = row["Min_LCOE Total Duration"]
  action_cost = row["Min_LCOE Action Cost"]
  labor_cost = row["Min_LCOE Labor Cost"]
  downtime_cost = row["Min_LCOE Downtime Cost"]
  ro_cost_increase = row["Min_LCOE RO Cost Increase"]
  total_cost = row["Min_LCOE Total Cost"]
  ro_subsys_uncertainties = row["Min_LCOE RO Subsys Uncertainties"]
  subsys_reliabilities = row["Min_LCOE Subsys Reliabilities"]
  reliability = row["Min_LCOE Reliability"]
  uptime = row["Min_LCOE Uptime"]
  energy_produced = row["Min_LCOE Energy"]
  lcoe = row["Min_LCOE LCOE"]
  # Initializing container for storing selected maintenance plan in binary format
  binary_plan = []
  # For each subsystem
  for i in range(M):
    # Initializing container for storing subsystem maintenance plan in binary format
    subsys_binary_plan = []
    # For each component in subsystem i
    for j in range(N[i]):
      # Initializing container for storing component maintenance plan in binary format
      comp_binary_plan = []
      # For each maintenance level
      for l in range(L):
        # If maintenance level l was selected
        if plan[i][j] == l:
          # Appending 1 to component maintenance plan to indicate repair action was selected
          comp_binary_plan.append(1)
        # Else
        else:
          # Appending 0 to component maintenance plan to indicate repair action was not selected
          comp_binary_plan.append(0)
      # Appending component maintenance plan to subsystem maintenance plan container
      subsys_binary_plan.append(comp_binary_plan)
    # Appending subsystem maintenance plan to complete maintenance plan container
    binary_plan.append(subsys_binary_plan)
  print(f"Binary plan: {binary_plan}")
  # Assessing selected maintenance plan with trial data
  num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe = solution_assessment_function(c_overtime, binary_plan, uptime, energy_produced, rho, LCOE_Durations_DF)
  # Storing results from trial testing in relevant dataframe
  LCOE_plan_analysis.loc[len(LCOE_plan_analysis)] = [rho, num_plans, comp_time, plan, nominal_duration, ro_duration_increase, total_duration, action_cost, labor_cost, downtime_cost, ro_cost_increase, total_cost, ro_subsys_uncertainties, subsys_reliabilities, reliability, uptime, energy_produced, lcoe, num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe]
  print(f"Assessing Min_Cost Plan...")
  # Extracting relevant data for cost-minimization maintenance plan
  # NOTE: Aside from changing what plan is being assessed and where data is being stored, this block of code functions the same as the one for LCOE-minimization.
  plan = row["Min_Cost Plan"]
  print(f"Plan: {plan}")
  nominal_duration = row["Min_Cost Nominal Duration"]
  ro_duration_increase = row["Min_Cost RO Duration Increase"]
  total_duration = row["Min_Cost Total Duration"]
  action_cost = row["Min_Cost Action Cost"]
  labor_cost = row["Min_Cost Labor Cost"]
  downtime_cost = row["Min_Cost Downtime Cost"]
  ro_cost_increase = row["Min_Cost RO Cost Increase"]
  total_cost = row["Min_Cost Total Cost"]
  ro_subsys_uncertainties = row["Min_Cost RO Subsys Uncertainties"]
  subsys_reliabilities = row["Min_Cost Subsys Reliabilities"]
  reliability = row["Min_Cost Reliability"]
  uptime = row["Min_Cost Uptime"]
  energy_produced = row["Min_Cost Energy"]
  lcoe = row["Min_Cost LCOE"]
  binary_plan = []
  for i in range(M):
    subsys_binary_plan = []
    for j in range(N[i]):
      comp_binary_plan = []
      for l in range(L):
        if plan[i][j] == l:
          comp_binary_plan.append(1)
        else:
          comp_binary_plan.append(0)
      subsys_binary_plan.append(comp_binary_plan)
    binary_plan.append(subsys_binary_plan)
  print(f"Binary plan: {binary_plan}")
  num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe = solution_assessment_function(c_overtime, binary_plan, uptime, energy_produced, rho, Cost_Durations_DF)
  Cost_plan_analysis.loc[len(Cost_plan_analysis)] = [rho, num_plans, comp_time, plan, nominal_duration, ro_duration_increase, total_duration, action_cost, labor_cost, downtime_cost, ro_cost_increase, total_cost, ro_subsys_uncertainties, subsys_reliabilities, reliability, uptime, energy_produced, lcoe, num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe]
  print(f"Assessing Max_Energy Plan...")
  # Extracting relevant data for energy-maximization maintenance plan
  # NOTE: Aside from changing what plan is being assessed and where data is being stored, this block of code functions the same as the one for LCOE-minimization.
  plan = row["Max_Energy Plan"]
  print(f"Plan: {plan}")
  nominal_duration = row["Max_Energy Nominal Duration"]
  ro_duration_increase = row["Max_Energy RO Duration Increase"]
  total_duration = row["Max_Energy Total Duration"]
  action_cost = row["Max_Energy Action Cost"]
  labor_cost = row["Max_Energy Labor Cost"]
  downtime_cost = row["Max_Energy Downtime Cost"]
  ro_cost_increase = row["Max_Energy RO Cost Increase"]
  total_cost = row["Max_Energy Total Cost"]
  ro_subsys_uncertainties = row["Max_Energy RO Subsys Uncertainties"]
  subsys_reliabilities = row["Max_Energy Subsys Reliabilities"]
  reliability = row["Max_Energy Reliability"]
  uptime = row["Max_Energy Uptime"]
  energy_produced = row["Max_Energy Energy"]
  lcoe = row["Max_Energy LCOE"]
  binary_plan = []
  for i in range(M):
    subsys_binary_plan = []
    for j in range(N[i]):
      comp_binary_plan = []
      for l in range(L):
        if plan[i][j] == l:
          comp_binary_plan.append(1)
        else:
          comp_binary_plan.append(0)
      subsys_binary_plan.append(comp_binary_plan)
    binary_plan.append(subsys_binary_plan)
  print(f"Binary plan: {binary_plan}")
  num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe = solution_assessment_function(c_overtime, binary_plan, uptime, energy_produced, rho, Energy_Durations_DF)
  Energy_plan_analysis.loc[len(Energy_plan_analysis)] = [rho, num_plans, comp_time, plan, nominal_duration, ro_duration_increase, total_duration, action_cost, labor_cost, downtime_cost, ro_cost_increase, total_cost, ro_subsys_uncertainties, subsys_reliabilities, reliability, uptime, energy_produced, lcoe, num_trials, break_duration_exceeded, break_duration_exceeded_proportion, budget_exceeded, budget_exceeded_proportion, avg_duration, std_dev_duration, avg_plan_duration_slack, avg_plan_ot, std_dev_plan_ot, avg_ot, std_dev_ot, avg_cost, std_dev_cost, avg_plan_cost_slack, avg_total_cost, std_dev_total_cost, avg_plan_ot_cost, std_dev_plan_ot_cost, avg_ot_cost, std_dev_ot_cost, avg_plan_budget_exceed, std_dev_plan_budget_exceed, avg_budget_exceed, std_dev_budget_exceed, avg_lcoe, std_dev_lcoe]
# Storing data from trial experiments in Excel workbooks
with pandas.ExcelWriter(file_path, engine='openpyxl') as writer:
  LCOE_plan_analysis.to_excel(writer, sheet_name=LCOE_sheet_name, index=False)
  Cost_plan_analysis.to_excel(writer, sheet_name=Cost_sheet_name, index=False)
  Energy_plan_analysis.to_excel(writer, sheet_name=Energy_sheet_name, index=False)
with pandas.ExcelWriter(file_path_durations, engine='openpyxl') as writer:
  LCOE_Durations_DF.to_excel(writer, sheet_name=LCOE_sheet_name, index=False)
  Cost_Durations_DF.to_excel(writer, sheet_name=Cost_sheet_name, index=False)
  Energy_Durations_DF.to_excel(writer, sheet_name=Energy_sheet_name, index=False)

# **Large System Testing**

## 21-Component Experimental Instance Parameters

In [ ]:
# Declaring relevant parameters for experimental instance
# Number of subsystems (M)
M = 7
# Number of components in each subsystem (N_i)
N = [4, 2, 3, 2, 3, 4, 3]
# Effective age of each component at beginning of maintenance break (B_ij)
Component_Age = [[60, 97, 68, 105],
                 [77, 71],
                 [100, 72, 73],
                 [103, 74],
                 [77, 104, 113, 81],
                 [10, 48, 39, 116],
                 [61, 92, 11]]
# Operational status of each component at beginning of maintenance break (S_ij)
Component_Status = [[1, 0, 0, 0],
                    [0, 1],
                    [1, 0, 1],
                    [0, 0],
                    [0, 1, 0],
                    [1, 1, 1, 0],
                    [1, 0, 0]]
# Scale parameters for component Weibull distributions (beta_ij)
Component_Weibull_eta = [[100, 100, 100, 100],
                         [160, 160],
                         [120, 120, 120],
                         [187, 187],
                         [80, 80, 80],
                         [100, 100, 100, 100],
                         [120, 120, 120]]
# Shape parameters for component Weibull distributions (eta_ij)
Component_Weibull_beta = [[1.5, 1.5, 1.5, 1.5],
                          [2.4, 2.4],
                          [1.8, 1.8, 1.8],
                          [2.8, 2.8],
                          [1.2, 1.2, 1.2],
                          [1.5, 1.5, 1.5, 1.5],
                          [1.8, 1.8, 1.8]]



# Number of maintenance levels under consideration for every component (L_ij)
L = 4
# Age reduction coefficient for maintenance level
Age_Reduc_Coeff = [1, 1, 0.25, 0]
# Maximum break duration (T_0)
T0 = 120
# Maintenance budget available (C_0)
C0 = 500
# Minimum reliability required (R_0)
R0 = 0.45
# Length of upcoming mission (W)
W = 60
# Variable cost per unit of time spent performing maintenance (c_v)
c_labor = 5
# Penalty cost per unit of system downtime (c_d)
c_downtime = 33
# Energy production rate function (epsilon(t))
energy_rate = 0.5
# Probability of maintenance occurring onshore (p)
p = 0.5
# Onshore maintenance cost factor (k_c)
k_cost = 1.25
# Onshore maintenance time factor (k_t)
k_time = 1.5



# Duration of corrective and preventive maintenance actions for each component while offshore (t_ijl^cm and t_ijl^pm)
# First dimension is subsystem, second is component, third is whether it's CM (index 0) or PM (index 1), fourth is level of maintenance
# This list must be adjusted based on the number of subsystems, number of components in each subsystem, and number of levels of maintenance
T_site = [[[[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]]],
          [[[0.00, 3.60, 5.40, 10.80],[0.00, 0.00, 2.70, 5.40]], [[0.00, 3.60, 5.40, 10.80],[0.00, 0.00, 2.70, 5.40]]],
          [[[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]]],
          [[[0.00, 4.00, 6.00, 12.00],[0.00, 0.00, 3.00, 6.00]], [[0.00, 4.00, 6.00, 12.00],[0.00, 0.00, 3.00, 6.00]]],
          [[[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]], [[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]], [[0.00, 2.40, 3.60, 7.20],[0.00, 0.00, 1.80, 3.60]]],
          [[[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]], [[0.00, 2.80, 4.20, 8.40],[0.00, 0.00, 2.10, 4.20]]],
          [[[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]], [[0.00, 3.20, 4.80, 9.60],[0.00, 0.00, 2.40, 4.80]]]]
# Fixed cost of corrective and preventive maintenance actions for each component while offshore (c_ijl^cm and c_ijl^pm)
# Same structure as T_site
C_site = [[[[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]]],
          [[[0.00, 15.00, 22.50, 45.00],[0.00, 0.00, 11.25, 22.50]], [[0.00, 15.00, 22.50, 45.00],[0.00, 0.00, 11.25, 22.50]]],
          [[[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]]],
          [[[0.00, 16.67, 25.00, 50.00],[0.00, 0.00, 12.50, 25.00]], [[0.00, 16.67, 25.00, 50.00],[0.00, 0.00, 12.50, 25.00]]],
          [[[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]], [[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]], [[0.00, 10.00, 15.00, 30.00],[0.00, 0.00, 7.50, 15.00]]],
          [[[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]], [[0.00, 11.67, 17.50, 35.00],[0.00, 0.00, 8.75, 17.50]]],
          [[[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]], [[0.00, 13.33, 20.00, 40.00],[0.00, 0.00, 10.00, 20.00]]]]



# Creating copy of offshore maintenance cost and duration data
C_expected = copy.deepcopy(C_site)
T_expected = copy.deepcopy(T_site)
# For each subsystem
for i in range(M):
  # For each component in subsystem i
  for j in range(N[i]):
    # For both lists storing corrective and preventive maintenance data
    for k in range(2):
      # Calculating expected cost and duration of each maintenance action (C_ijl and T_ijl) based on onshore cost and time factors (k_c and k_t) and probability of onshore work (p)
      C_expected[i][j][k] = [l*(k_cost*p + 1 - p) for l in C_expected[i][j][k]]
      T_expected[i][j][k] = [l*(k_time*p + 1 - p) for l in T_expected[i][j][k]]

## *Sensitivity Analysis*

### Headers

In [ ]:
# Listing headers to be used for storing data from experimental testing
headers = ["Parameter", "Number of Feasible Maintenance Plans", "CPU Time",
           "Min_LCOE Plan", "Min_LCOE Duration", "Min_LCOE Action Cost", "Min_LCOE Labor Cost", "Min_LCOE Downtime Cost", "Min_LCOE Total Cost", "Min_LCOE Subsys Reliabilities", "Min_LCOE Reliability", "Min_LCOE Uptime", "Min_LCOE Energy", "Min_LCOE LCOE",
           "Min_Cost Plan", "Min_Cost Duration", "Min_Cost Action Cost", "Min_Cost Labor Cost", "Min_Cost Downtime Cost", "Min_Cost Total Cost", "Min_Cost Subsys Reliabilities", "Min_Cost Reliability", "Min_Cost Uptime", "Min_Cost Energy", "Min_Cost LCOE",
           "Max_Energy Plan", "Max_Energy Duration", "Max_Energy Action Cost", "Max_Energy Labor Cost", "Max_Energy Downtime Cost", "Max_Energy Total Cost", "Max_Energy Subsys Reliabilities", "Max_Energy Reliability", "Max_Energy Uptime", "Max_Energy Energy", "Max_Energy LCOE"]

### Downtime Cost

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.45
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(33, 34, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value
  Outcome_tcr = deterministic_variant_tcr()
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic_21Comps.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}.")

# **Alternative Policy Testing**

## *Subfunctions for Subsystem Plan Generation Under Different Policies*

In [ ]:
# Defining function for generating list of maintenance actions available for each component in subsystem i under "mR Only" policy
# Inputs: subsystem under consideration
# Outputs: list of lists containing maintenance actions available for each component in subsystem under consideration
def avail_maint_list_mr(i):
  # Initializing list for tracking maintenance actions available for each component
  Avail_Maint = []

  # For each component in subsystem i
  for j in range(N[i]):
    # Generating list of all possible maintenance actions for component (DN and mR only)
    Comp_Maint = [0, 1]

    # If component is functioning
    if Component_Status[i][j] == 1:
      # Removing minimal repair action
      Comp_Maint.pop(1)

    # Appending list of maintenance actions for component to list for subsystem
    Avail_Maint.append(Comp_Maint)

  # Returning outputs
  return Avail_Maint



# Defining function for generating list of maintenance actions available for each component in subsystem i under "IR Only" policy
# Inputs: subsystem under consideration
# Outputs: list of lists containing maintenance actions available for each component in subsystem under consideration
def avail_maint_list_ir(i):
  # Initializing list for tracking maintenance actions available for each component
  Avail_Maint = []

  # For each component in subsystem i
  for j in range(N[i]):
    # Generating list of all possible maintenance actions for component (DN and IR only)
    Comp_Maint = [0, 2]
    # Appending list of maintenance actions for component to list for subsystem
    Avail_Maint.append(Comp_Maint)

  # Returning outputs
  return Avail_Maint



# Defining function for generating list of maintenance actions available for each component in subsystem i under "PR Only" policy
# Inputs: subsystem under consideration
# Outputs: list of lists containing maintenance actions available for each component in subsystem under consideration
def avail_maint_list_pr(i):
  # Initializing list for tracking maintenance actions available for each component
  Avail_Maint = []

  # For each component in subsystem i
  for j in range(N[i]):
    # Generating list of all possible maintenance actions for component (DN and PR only)
    Comp_Maint = [0, 3]
    # Appending list of maintenance actions for component to list for subsystem
    Avail_Maint.append(Comp_Maint)

  # Returning outputs
  return Avail_Maint



# Defining function for generating list of maintenance actions available for each component in subsystem i under "CM Only" policy
# Inputs: subsystem under consideration
# Outputs: list of lists containing maintenance actions available for each component in subsystem under consideration
def avail_maint_list_cm(i):
  # Initializing list for tracking maintenance actions available for each component
  Avail_Maint = []

  # For each component in subsystem i
  for j in range(N[i]):
    # Generating list of all possible maintenance actions for component
    Comp_Maint = list(range(0, L))

    # If component is functioning
    if Component_Status[i][j] == 1:
      # Removing all repair actions (aside from DN)
      Comp_Maint = [0]

    # Appending list of maintenance actions for component to list for subsystem
    Avail_Maint.append(Comp_Maint)

  # Returning outputs
  return Avail_Maint

## *Subsystem Plan Generation Function*

In [ ]:
# Defining function for generating complete, sorted, culled list of maintenance plans for each subsystem
# Inputs: policy selected for maintenance planning ("mR Only", "IR Only", "PR Only", "CM Only")
# NOTE: If left blank, then defaults to SM policy
# Outputs: 1) complete, sorted list of feasible maintenance plans for each subsystem, as well as sorted lists reporting each plan's respective 2) duration, 3) maintenance cost,
# 4) subsystem conditional reliability, and 5) total cost (maintenance + labour)
def subsystem_solution_generator_tcr(policy):
  # Initializing lists for storing subsystem maintenance plans (Lambda_i) and respective durations (T_i^lambda), repair costs, conditional reliabilities (R_i^clambda(W)), and total costs (C_i^lambda)
  all_subsys_solns = []
  all_subsys_soln_durations = []
  all_subsys_soln_costs = []
  all_subsys_soln_reliabilities = []
  all_subsys_soln_TOTAL_costs = []

  # For each subsystem
  for i in range(M):

    # If "mR Only" policy is selected
    if policy == "mR":
      print("mR Policy Selected...")
      # Generating list of available maintenance actions for each component in subsystem i under "mR Only" policy
      Avail_Maint = avail_maint_list_mr(i)
    # Else, if "IR Only" policy is selected
    elif policy == "IR":
      print("IR Policy Selected...")
      # Generating list of available maintenance actions for each component in subsystem i under "IR Only" policy
      Avail_Maint = avail_maint_list_ir(i)
    # Else, if "PR Only" policy is selected
    elif policy == "PR":
      print("PR Policy Selected...")
      # Generating list of available maintenance actions for each component in subsystem i under "PR Only" policy
      Avail_Maint = avail_maint_list_pr(i)
    # Else, if "CM Only" policy is selected
    elif policy == "CM":
      print("CM Policy Selected...")
      # Generating list of available maintenance actions for each component in subsystem i under "CM Only" policy
      Avail_Maint = avail_maint_list_cm(i)
    # Else
    else:
      print("No Policy Selected...")
      # Generating list of available maintenance actions for each component in subsystem i under original SM policy
      Avail_Maint = avail_maint_list(i)

    # Generating list of all possible maintenance plans for subsystem i
    subsys_solns = solution_enumeration(Avail_Maint)
    # Initializing lists for tracking maintenance plan data for subsystem i
    subsys_soln_durations = []
    subsys_soln_costs = []
    subsys_soln_reliabilities = []
    subsys_soln_TOTAL_costs = []
    # Initializing counter for scanning through potential subsystem i maintenance plans
    s = 0

    # While still not done scanning through subsystem i maintenance plans
    while s < len(subsys_solns):
      # Initializing subsystem i maintenance plan duration, repair cost, and conditional failure
      Subsys_Soln_Duration = 0
      Subsys_Soln_Cost = 0
      Subsys_Soln_Failure = 1

      # For each component in subsystem i
      for j in range(N[i]):
        # Adding component E_ij maintenance action duration (T_ijl) and repair cost to total subsystem i plan duration and repair cost
        Subsys_Soln_Duration += T_expected[i][j][Component_Status[i][j]][subsys_solns[s][j]]
        Subsys_Soln_Cost += C_expected[i][j][Component_Status[i][j]][subsys_solns[s][j]]
        # Calculating updated component E_ij effective age and operational status (A_ij and V_ij) based on selected maintenance action
        A_ij = Component_Age[i][j]*Age_Reduc_Coeff[subsys_solns[s][j]]
        V_ij = 0

        # If component E_ij was functional before maintenance or has received a maintenance action other than "Do Nothing"
        if (Component_Status[i][j] == 1) or (subsys_solns[s][j] != 0):
          # Setting component operational status to 1 (functional)
          V_ij = 1

        # Calculating component E_ij conditional reliability (R_ij^c(W)) based on selected maintenance action
        R_ij = V_ij*math.exp(-((A_ij + W)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j])/math.exp(-((A_ij)/Component_Weibull_eta[i][j])**Component_Weibull_beta[i][j])
        # Factoring in component E_ij conditional reliability to subsystem i maintenance plan conditional failure (1 - R_i^c(W))
        Subsys_Soln_Failure = Subsys_Soln_Failure*(1 - R_ij)

      # Calculating subsystem i conditional reliability according to conditional failure of plan (R_i^c(W))
      Subsys_Soln_Reliability = 1 - Subsys_Soln_Failure

      # If subsystem i maintenance plan duration exceeds break duration, total cost exceeds maintenance budget, or conditional reliability is below minimum threshold
      if Subsys_Soln_Duration > T0 or (Subsys_Soln_Cost + Subsys_Soln_Duration*c_labor) > C0 or Subsys_Soln_Reliability < R0:
        # Removing subsystem i maintenance plan
        subsys_solns.pop(s)
        # Decrementing counter for scanning through subsystem i maintenance plans since a plan has been removed
        s -= 1
      # Else
      else:
        # Appending plan's duration, repair cost, conditional reliability, and total cost to overall lists for subsystem i maintenance plans
        subsys_soln_durations.append(Subsys_Soln_Duration)
        subsys_soln_costs.append(Subsys_Soln_Cost)
        subsys_soln_reliabilities.append(Subsys_Soln_Reliability)
        subsys_soln_TOTAL_costs.append(Subsys_Soln_Cost + Subsys_Soln_Duration*c_labor)

      # Incrementing counter for scanning through subsystem i maintenance plans to move onto next plan
      s += 1

    # If no feasible subsystem plans were found
    if len(subsys_solns) == 0:
      # Reporting that there are no feasible solutions to the problem based on its current parameters
      print("No feasible solutions based on given parameters.")
      # Appending empty lists of feasible subsystem i maintenance plans and associated data to overall lists
      all_subsys_solns.append(subsys_solns)
      all_subsys_soln_durations.append(subsys_soln_durations)
      all_subsys_soln_costs.append(subsys_soln_costs)
      all_subsys_soln_reliabilities.append(subsys_soln_reliabilities)
      all_subsys_soln_TOTAL_costs.append(subsys_soln_TOTAL_costs)
      # Ending subsystem maintenance plan generation
      break

    # Sorting feasible subsystem i maintenance plans based on durations in increasing order
    subsys_soln_durations, subsys_solns, subsys_soln_costs, subsys_soln_reliabilities, subsys_soln_TOTAL_costs = zip(*sorted(zip(subsys_soln_durations, subsys_solns, subsys_soln_costs, subsys_soln_reliabilities, subsys_soln_TOTAL_costs)))
    # Converting sorted lists of subsystem i maintenance plans and associated data from tuples to lists
    subsys_solns = list(subsys_solns)
    subsys_soln_durations = list(subsys_soln_durations)
    subsys_soln_costs = list(subsys_soln_costs)
    subsys_soln_reliabilities = list(subsys_soln_reliabilities)
    subsys_soln_TOTAL_costs = list(subsys_soln_TOTAL_costs)
    # Appending lists of feasible, sorted subsystem i maintenance plans and associated data to overall list (Lambda_i, T_i^lambda, R_i^clambda(W), C_i^lambda)
    all_subsys_solns.append(subsys_solns)
    all_subsys_soln_durations.append(subsys_soln_durations)
    all_subsys_soln_costs.append(subsys_soln_costs)
    all_subsys_soln_reliabilities.append(subsys_soln_reliabilities)
    all_subsys_soln_TOTAL_costs.append(subsys_soln_TOTAL_costs)

  # Returning outputs
  return all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, all_subsys_soln_TOTAL_costs

## *Function for Generating Complete, Feasible Maintenance Plans (Under Different Policies)*

In [ ]:
# Defining master function for constructing all feasible, complete maintenance plans
# Inputs: policy selected for maintenance planning ("mR Only", "IR Only", "PR Only", "CM Only")
# NOTE: If left blank, then defaults to SM policy
# Outputs: 1) list of all feasible, complete maintenance plans and their associated 2) durations, 3) repair costs, 4) conditional system reliabilities, and 5) conditional subsystem reliabilities
def plan_list_generator_time_cost_reliability_constraints(policy):
  # Initializing lists for storing all feasible, complete maintenance plans and associated data (Theta)
  complete_maint_plan_master_list = []
  complete_maint_plan_duration_master_list = []
  complete_maint_plan_cost_master_list = []
  complete_maint_plan_reliability_master_list = []
  complete_maint_plan_subsys_reliability_master_list = []

  # If the system only has one subsystem
  if M == 1:
    # Generating all feasible maintenance plans for the subsystem under selected policy
    complete_maint_plans, complete_maint_plan_durations, complete_maint_plan_costs, complete_maint_plan_reliabilities, complete_maint_plan_TOTAL_costs = subsystem_solution_generator_tcr(policy)
    # Storing all feasible maintenance plans and associated data for the subsystem as the complete maintenance plan results
    complete_maint_plan_master_list = complete_maint_plans[0]
    complete_maint_plan_duration_master_list = complete_maint_plan_durations[0]
    complete_maint_plan_cost_master_list = complete_maint_plan_costs[0]
    complete_maint_plan_reliability_master_list = complete_maint_plan_reliabilities[0]
  # Else
  else:
    # Generating all feasible maintenance plans and associated data for all subsystems under selected policy
    all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, all_subsys_soln_TOTAL_costs = subsystem_solution_generator_tcr(policy)

    # For each subsystem
    for i in range(len(all_subsys_solns)):

      # If there are no feasible plans for the subsystem
      if len(all_subsys_solns[i]) == 0:
        # Returning the empty lists of complete maintenance plans and associated data (i.e., there are no feasible solutions)
        return complete_maint_plan_master_list, complete_maint_plan_duration_master_list, complete_maint_plan_cost_master_list, complete_maint_plan_reliability_master_list, complete_maint_plan_subsys_reliability_master_list

    # Isolating the feasible maintenance plans and associated data for subsystem 1
    subsys_1_solns = all_subsys_solns[0]
    subsys_1_soln_durations = all_subsys_soln_durations[0]
    subsys_1_soln_costs = all_subsys_soln_costs[0]
    subsys_1_soln_reliabilities = all_subsys_soln_reliabilities[0]
    subsys_1_soln_TOTAL_costs = all_subsys_soln_TOTAL_costs[0]
    # Copying the feasible maintenance plans and associated data for subsystem 2
    subsys_2_solns = copy.deepcopy(all_subsys_solns[1])
    subsys_2_soln_durations = copy.deepcopy(all_subsys_soln_durations[1])
    subsys_2_soln_costs = copy.deepcopy(all_subsys_soln_costs[1])
    subsys_2_soln_reliabilities = copy.deepcopy(all_subsys_soln_reliabilities[1])
    subsys_2_soln_TOTAL_costs = copy.deepcopy(all_subsys_soln_TOTAL_costs[1])

    # For each potential maintenance plan for subsystem 1 (Lambda_1)
    for soln in range(len(subsys_1_solns)):
      # Initializing the variable tracking the current subsystem being considered for building maintenance plans
      m = 1
      # Using the potential maintenance plan and assoicated data for subsystem 1 as the basis for a complete maintenance plan (Plan_next, T_next, R_next, C_next)
      current_plan = [subsys_1_solns[soln]]
      current_plan_duration = subsys_1_soln_durations[soln]
      current_plan_cost = subsys_1_soln_costs[soln]
      current_plan_reliability = subsys_1_soln_reliabilities[soln]
      current_plan_TOTAL_cost = subsys_1_soln_TOTAL_costs[soln]
      ##### Break Duration Constraint #####
      # Calculating how much time remains to be allocated to remaining subsystem maintenance plans (T_0 - T_next)
      remaining_duration = T0 - current_plan_duration
      # Finding the index of the first infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the remaining time
      # (NOTE: the lists are already sorted from lowest to highest durations)
      first_infeasible_ss2_plan = bisect.bisect(subsys_2_soln_durations, remaining_duration)
      # Removing all subsystem 2 plans and associated data with an index equal to or higher than the first infeasible plan's
      subsys_2_solns = subsys_2_solns[:first_infeasible_ss2_plan]
      subsys_2_soln_durations = subsys_2_soln_durations[:first_infeasible_ss2_plan]
      subsys_2_soln_costs = subsys_2_soln_costs[:first_infeasible_ss2_plan]
      subsys_2_soln_reliabilities = subsys_2_soln_reliabilities[:first_infeasible_ss2_plan]
      subsys_2_soln_TOTAL_costs = subsys_2_soln_TOTAL_costs[:first_infeasible_ss2_plan]

      # If there is at least one subsystem 2 plan that has remained (is feasible based on the subsystem 1 plan that was selected)
      if(len(subsys_2_solns) > 0):
        ##### Maintenance Budget Constraint #####
        # Copying the remaining feasible plans and associated data for the next subsystem to be considered (Lambda_2^copy)
        subsys_2_solns_costrelcopy = copy.deepcopy(subsys_2_solns)
        subsys_2_soln_durations_costrelcopy = copy.deepcopy(subsys_2_soln_durations)
        subsys_2_soln_costs_costrelcopy = copy.deepcopy(subsys_2_soln_costs)
        subsys_2_soln_reliabilities_costrelcopy = copy.deepcopy(subsys_2_soln_reliabilities)
        subsys_2_soln_TOTAL_costs_costrelcopy = copy.deepcopy(subsys_2_soln_TOTAL_costs)
        # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on total costs
        subsys_2_soln_TOTAL_costs_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_reliabilities_costrelcopy = zip(*sorted(zip(subsys_2_soln_TOTAL_costs_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_reliabilities_costrelcopy)))
        subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
        subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
        subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
        subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
        subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
        # Calculating how much budget remains to be allocated to remaining subsystem maintenance plans (C_0 - C_next)
        remaining_budget = C0 - current_plan_TOTAL_cost
        # Finding the index of the first infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the remaining budget
        first_infeasible_next_ss_plan = bisect.bisect(subsys_2_soln_TOTAL_costs_costrelcopy, remaining_budget)
        # Removing all subsystem 2 plans and associated data with an index equal to or higher than the first infeasible plan's
        subsys_2_solns_costrelcopy = subsys_2_solns_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_durations_costrelcopy = subsys_2_soln_durations_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_costs_costrelcopy = subsys_2_soln_costs_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_reliabilities_costrelcopy = subsys_2_soln_reliabilities_costrelcopy[:first_infeasible_next_ss_plan]
        subsys_2_soln_TOTAL_costs_costrelcopy = subsys_2_soln_TOTAL_costs_costrelcopy[:first_infeasible_next_ss_plan]

        # If there is at least one subsystem 2 plan that has remained (is feasible based on the subsystem 1 plan that was selected)
        if(len(subsys_2_solns_costrelcopy) > 0):
          ##### Minimum Reliability Constraint #####
          # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on conditional reliabilities
          subsys_2_soln_reliabilities_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy = zip(*sorted(zip(subsys_2_soln_reliabilities_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy)))
          subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
          subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
          subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
          subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
          subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
          # Calculating minimum conditional reliabilty that must be met by each remaining subsystem (R_0 - R_next)
          minimum_reliability = R0/current_plan_reliability
          # Finding the index of the last infeasible maintenance plan in the list of plans for subsystem 2 to be considered based on the minimum conditional reliability
          first_infeasible_next_ss_plan = bisect.bisect_left(subsys_2_soln_reliabilities_costrelcopy, minimum_reliability)
          # Removing all subsystem 2 plans and associated data with an index equal to or less than the last infeasible plan's
          subsys_2_solns_costrelcopy = subsys_2_solns_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_durations_costrelcopy = subsys_2_soln_durations_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_costs_costrelcopy = subsys_2_soln_costs_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_reliabilities_costrelcopy = subsys_2_soln_reliabilities_costrelcopy[first_infeasible_next_ss_plan:]
          subsys_2_soln_TOTAL_costs_costrelcopy = subsys_2_soln_TOTAL_costs_costrelcopy[first_infeasible_next_ss_plan:]

          # If there is at least one subsystem 2 plan that has remained (is feasible based on the current partial plan)
          if len(subsys_2_solns_costrelcopy) > 0:
            # Re-sorting lists of feasible plans and associated data for subsystem 2 to be considered to be increasing based on total durations
            subsys_2_soln_durations_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy = zip(*sorted(zip(subsys_2_soln_durations_costrelcopy, subsys_2_solns_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, subsys_2_soln_TOTAL_costs_costrelcopy)))
            subsys_2_solns_costrelcopy = list(subsys_2_solns_costrelcopy)
            subsys_2_soln_durations_costrelcopy = list(subsys_2_soln_durations_costrelcopy)
            subsys_2_soln_costs_costrelcopy = list(subsys_2_soln_costs_costrelcopy)
            subsys_2_soln_reliabilities_costrelcopy = list(subsys_2_soln_reliabilities_costrelcopy)
            subsys_2_soln_TOTAL_costs_costrelcopy = list(subsys_2_soln_TOTAL_costs_costrelcopy)
            # Calling the recursive plan builder subfunction with selected subsystem 1 plan and remaining feasible plans for subsystem 2
            complete_maint_plans, complete_maint_plan_durations, complete_maint_plan_costs, complete_maint_plan_reliabilities = plan_builder_tcr(all_subsys_solns, all_subsys_soln_durations, all_subsys_soln_costs, all_subsys_soln_reliabilities, current_plan, current_plan_duration, current_plan_cost, current_plan_reliability, subsys_2_solns_costrelcopy, subsys_2_soln_durations_costrelcopy, subsys_2_soln_costs_costrelcopy, subsys_2_soln_reliabilities_costrelcopy, m, all_subsys_soln_TOTAL_costs, current_plan_TOTAL_cost, subsys_2_soln_TOTAL_costs_costrelcopy)

            # For all complete, feasible maintenance plans constructed from the plan builder subfunction
            for n in range(len(complete_maint_plans)):
              # Initializing list to store subsystem conditional reliabilities for selected plan
              current_maint_plan_subsys_reliabilities = []

              # For each subsystem
              for i in range(M):
                # Finding the index of the corresponding subsystem maintenance plan and storing the associated subsystem conditional reliability
                subsys_plan_index = all_subsys_solns[i].index(complete_maint_plans[n][i])
                current_maint_plan_subsys_reliabilities.append(all_subsys_soln_reliabilities[i][subsys_plan_index])

              # Appending the list of subsystem conditional reliabilities for the selected complete maintenance plan
              complete_maint_plan_subsys_reliability_master_list.append(current_maint_plan_subsys_reliabilities)

            # Storing the complete, feasible maintenance plans that were constructed and their associated data in the master lists
            complete_maint_plan_master_list += complete_maint_plans
            complete_maint_plan_duration_master_list += complete_maint_plan_durations
            complete_maint_plan_cost_master_list += complete_maint_plan_costs
            complete_maint_plan_reliability_master_list += complete_maint_plan_reliabilities

  # Returning outputs
  return complete_maint_plan_master_list, complete_maint_plan_duration_master_list, complete_maint_plan_cost_master_list, complete_maint_plan_reliability_master_list, complete_maint_plan_subsys_reliability_master_list

## *Deterministic MRE SMP Solution Methodology Master Function (Under Different Policies)*

In [ ]:
# Defining master function for finding optimal maintenance plan
# Inputs: policy selected for maintenance planning ("mR Only", "IR Only", "PR Only", "CM Only")
# NOTE: If left blank, then defaults to SM policy
# Outputs: 1) number of feasible solutions, 2) computation time,
# For LCOE minimization: 3) optimal maintenance plan and associated 4) duration, 5) repair cost, 6) labour cost, 7) downtime cost, 8) total cost, 9) subsystem conditional reliabilities, 10) system conditional reliability, 11) expected uptime, 12) expected energy produced, and 13) expected LCOE
# For cost minimization: 14) optimal maintenance plan and associated 15) duration, 16) repair cost, 17) labour cost, 18) downtime cost, 19) total cost, 20) subsystem conditional reliabilities, 21) system conditional reliability, 22) expected uptime, 23) expected energy produced, and 24) expected LCOE
# For energy production maximization: 25) optimal maintenance plan and associated 26) duration, 27) repair cost, 28) labour cost, 29) downtime cost, 30) total cost, 31) subsystem conditional reliabilities, 32) system conditional reliability, 33) expected uptime, 34) expected energy produced, and 35) expected LCOE
def deterministic_variant_tcr(policy):
  # Recording start time of experiment
  overall_start_time = time.time()
  print(f"Generating plans with time, cost, and reliability constraints (T_0 = {T0}, C_0 = {C0}, R_0 = {R0})...")
  # Recording start time of feasible plan generation
  soln_generation_start_time = time.time()
  # Generating all complete, feasible maintenance plans based on given parameters
  maint_plan_list, plan_duration_list, plan_cost_list, plan_reliability_list, plan_subsys_reliability_list = plan_list_generator_time_cost_reliability_constraints(policy)
  # Calculating and reporting time spent generating maintenance plans
  soln_generation_end_time = time.time()
  print(f"{len(maint_plan_list)} plans generated!")
  print(f"Solution generation took {soln_generation_end_time - soln_generation_start_time} seconds.")

  # If no feasible maintenance plans were constructed
  if len(maint_plan_list) == 0:
    # Calculating total experiment time and reporting infeasibility
    overall_end_time = time.time()
    total_time = overall_end_time - overall_start_time
    print("No feasible solutions found.")
    # Constructing surrogate data for denoting infeasible experiment
    non_results = [len(maint_plan_list), total_time] + ["N/A"]*(len(headers) - 3)
    # Returning surrogate data
    return non_results

  # Initializing list for storing data needed to calculate expected LCOE for each feasible maintenance plan
  LCOE_data_list = []

  # For each feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Storing data needed to calculate expected LCOE for given maintenance plan
    LCOE_data_list.append((maint_plan_list[soln], plan_duration_list[soln], plan_cost_list[soln]))

  print("Now calculating uptime for all feasible solutions.")
  # Recording start time of uptime and LCOE calculations for feasible maintenance plans
  uptime_start_time = time.time()

  # Setting up parallelization process to allow uptime and LCOE to be calculate for multiple maintenance plans simultaneously
  if __name__ == "__main__":
      with multiprocessing.Pool() as pool:
          results = pool.map(expected_uptime, LCOE_data_list)

  # Calculating and reporting total uptime/LCOE calculation time
  uptime_end_time = time.time()
  print(f"Uptimes calculated in {uptime_end_time - uptime_start_time} seconds.")
  # Initializing lists for storing uptime, total cost, energy produced, and LCOE data for each feasible maintenance plan
  plan_uptime_list = []
  plan_TotalCost_list = []
  plan_EnergyProd_list = []
  plan_LCOE_list = []

  # For each feasible maintenance plan
  for soln in range(len(maint_plan_list)):
    # Saving expected uptime, total cost, energy produced, and LCOE
    plan_uptime_list.append(results[soln][0])
    plan_TotalCost_list.append(results[soln][1])
    plan_EnergyProd_list.append(results[soln][2])
    plan_LCOE_list.append(results[soln][3])

  # Finding index of feasible maintenance plans with lowest expected LCOE, total cost, and energy produced
  min_LCOE_index = plan_LCOE_list.index(min(plan_LCOE_list))
  min_cost_index = plan_TotalCost_list.index(min(plan_TotalCost_list))
  max_energy_prod_index = plan_EnergyProd_list.index(max(plan_EnergyProd_list))
  # Calculating and reporting total time spent performing optimization experiment
  overall_end_time = time.time()
  total_time = overall_end_time - overall_start_time
  print(f"\nTotal time to execute code: {total_time} seconds.")
  # Returning outputs
  return [len(maint_plan_list), total_time,
          maint_plan_list[min_LCOE_index], plan_duration_list[min_LCOE_index], plan_cost_list[min_LCOE_index], c_labor*plan_duration_list[min_LCOE_index], c_downtime*(W - plan_uptime_list[min_LCOE_index]), plan_TotalCost_list[min_LCOE_index], plan_subsys_reliability_list[min_LCOE_index], plan_reliability_list[min_LCOE_index], plan_uptime_list[min_LCOE_index], plan_EnergyProd_list[min_LCOE_index], plan_LCOE_list[min_LCOE_index],
          maint_plan_list[min_cost_index], plan_duration_list[min_cost_index], plan_cost_list[min_cost_index], c_labor*plan_duration_list[min_cost_index], c_downtime*(W - plan_uptime_list[min_cost_index]), plan_TotalCost_list[min_cost_index], plan_subsys_reliability_list[min_cost_index], plan_reliability_list[min_cost_index], plan_uptime_list[min_cost_index], plan_EnergyProd_list[min_cost_index], plan_LCOE_list[min_cost_index],
          maint_plan_list[max_energy_prod_index], plan_duration_list[max_energy_prod_index], plan_cost_list[max_energy_prod_index], c_labor*plan_duration_list[max_energy_prod_index], c_downtime*(W - plan_uptime_list[max_energy_prod_index]), plan_TotalCost_list[max_energy_prod_index], plan_subsys_reliability_list[max_energy_prod_index], plan_reliability_list[max_energy_prod_index], plan_uptime_list[max_energy_prod_index], plan_EnergyProd_list[max_energy_prod_index], plan_LCOE_list[max_energy_prod_index]]

## *Sensitivity Analysis*

### Headers

In [ ]:
# Listing headers to be used for storing data from experimental testing
headers = ["Parameter", "Number of Feasible Maintenance Plans", "CPU Time",
           "Min_LCOE Plan", "Min_LCOE Duration", "Min_LCOE Action Cost", "Min_LCOE Labor Cost", "Min_LCOE Downtime Cost", "Min_LCOE Total Cost", "Min_LCOE Subsys Reliabilities", "Min_LCOE Reliability", "Min_LCOE Uptime", "Min_LCOE Energy", "Min_LCOE LCOE",
           "Min_Cost Plan", "Min_Cost Duration", "Min_Cost Action Cost", "Min_Cost Labor Cost", "Min_Cost Downtime Cost", "Min_Cost Total Cost", "Min_Cost Subsys Reliabilities", "Min_Cost Reliability", "Min_Cost Uptime", "Min_Cost Energy", "Min_Cost LCOE",
           "Max_Energy Plan", "Max_Energy Duration", "Max_Energy Action Cost", "Max_Energy Labor Cost", "Max_Energy Downtime Cost", "Max_Energy Total Cost", "Max_Energy Subsys Reliabilities", "Max_Energy Reliability", "Max_Energy Uptime", "Max_Energy Energy", "Max_Energy LCOE"]

### Downtime Cost -- mR (Unrestricted $R_0$)

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.0000001
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5
policy = "mR"



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(33, 34, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value under "mR Only" policy
  Outcome_tcr = deterministic_variant_tcr(policy)
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic_mR.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}. Now downloading!")

### Downtime Cost -- IR

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.0000001
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5
policy = "IR"



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(33, 34, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value under "IR Only" policy
  Outcome_tcr = deterministic_variant_tcr(policy)
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic_IR.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}. Now downloading!")

### Downtime Cost -- PR (Unrestricted $R_0$)

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.0000001
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5
policy = "PR"



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(33, 34, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value under "PR Only" policy
  Outcome_tcr = deterministic_variant_tcr(policy)
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic_PR.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}. Now downloading!")

### Downtime Cost -- CM

In [ ]:
# Listing default parameter values
T0 = 120
C0 = 500
R0 = 0.0000001
W = 60
c_labor = 5
c_downtime = 33
p = 0.5
k_cost = 1.25
k_time = 1.5
policy = "CM"



# Generating list of values to be tested for selected parameter in sensitivity analysis
Parameter_value_list = list(numpy.arange(33, 34, 1))
# Initializing list of results from sensitivity analysis
Results_tcr = []
# Declaring name of parameter being varied in sensitivity analysis
headers[0] = "Downtime Cost (c_downtime)"

# For each value to be tested in sensitivity analysis
for l in range(len(Parameter_value_list)):
  # Replacing parameter default value with selected value to be tested
  c_downtime = Parameter_value_list[l]
  print(f"Testing with c_downtime = {c_downtime}...")
  # Performing optimization experiment with changed parameter value under "CM Only" policy
  Outcome_tcr = deterministic_variant_tcr(policy)
  # Storing solution results from experiment
  Results_tcr.append([c_downtime] + Outcome_tcr)

# Transforming data from sensitivity analysis experiments into dataframe
df_tcr = pandas.DataFrame(Results_tcr, columns=headers)
# Storing data from sensitivity analysis experiments in Excel workbook
output_path = "output_DowntimeCost_Deterministic_CM.xlsx"
with pandas.ExcelWriter(output_path, engine='openpyxl') as writer:
  df_tcr.to_excel(writer, sheet_name='Time, Cost, and Rel', index=False)
print(f"Data saved to {output_path}. Now downloading!")